# Step3 산불발생 선행기상 및 국지임계치 심화분석

노트북은 코드 실행과 산출물 생성만 담당한다. 해석은 `Step3_산불발생_선행기상및국지임계치_심화분석_진행예정로그.md`에 기록한다.


## S3-01 : 입력·파생·누수 감사

Step2에서 확정한 산불 고유 노출과 고정 매칭 대조군을 그대로 읽고, 모든 선행 시간창이 기준시각 현재 값을 제외하는지 감사한다.


In [2]:
from pathlib import Path
import hashlib
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

warnings.filterwarnings("ignore")

plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.dpi"] = 180
font_path = "C:/Windows/Fonts/malgun.ttf"
try:
    fm.fontManager.addfont(font_path)
    font_name = fm.FontProperties(fname=font_path).get_name()
except Exception:
    font_name = "Malgun Gothic"
sns.set_theme(
    style="whitegrid",
    font=font_name,
    rc={
        "font.family": font_name,
        "font.sans-serif": [font_name],
        "axes.unicode_minus": False,
    },
)
plt.rcParams["font.family"] = font_name
plt.rcParams["font.sans-serif"] = [font_name]
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", None)

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = next((p for p in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (p / "data" / "학습데이터").exists()), Path(r"D:/farm-system-public-02"))
DATA_DIR = REPO_ROOT / "data" / "학습데이터"
STEP2_TABLE_DIR = REPO_ROOT / "jsw" / "강원_재_EDA" / "outputs" / "Step2" / "tables"
STEP3_ROOT = REPO_ROOT / "jsw" / "강원_재_EDA" / "outputs" / "Step3"
TABLE_DIR = STEP3_ROOT / "tables"
PLOT_DIR = STEP3_ROOT / "plots"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

S3_ID = "S3-01"
WINDOW_HOURS = [1, 3, 6, 12, 24, 48, 72]
ROLLING_RECALC_SPECS = [
    ("직전24h_평균습도", "시점_습도_pct", 24, "mean"),
    ("직전24h_최소습도", "시점_습도_pct", 24, "min"),
    ("직전48h_평균습도", "시점_습도_pct", 48, "mean"),
    ("직전48h_최소습도", "시점_습도_pct", 48, "min"),
    ("직전24h_평균풍속", "시점_풍속_m_s", 24, "mean"),
    ("직전24h_최대풍속", "시점_풍속_m_s", 24, "max"),
    ("직전48h_평균풍속", "시점_풍속_m_s", 48, "mean"),
    ("직전48h_최대풍속", "시점_풍속_m_s", 48, "max"),
]


def read_csv_kr(path, **kwargs):
    return pd.read_csv(path, encoding="utf-8-sig", **kwargs)


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def write_table(df, name, **kwargs):
    path = TABLE_DIR / name
    df.to_csv(path, index=False, encoding="utf-8-sig", **kwargs)
    return path


def save_current_plot(name):
    path = PLOT_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    return path

print("REPO_ROOT", REPO_ROOT)
print("STEP2_TABLE_DIR", STEP2_TABLE_DIR)
print("TABLE_DIR", TABLE_DIR)
print("PLOT_DIR", PLOT_DIR)


In [3]:
# S3-01 입력: Step 2에서 확정한 산불 노출·고정 매칭 대조군만 사용한다.
paths = {
    "fire_unique": STEP2_TABLE_DIR / "S2-01_unique_weather_exposures.csv",
    "matched_controls": STEP2_TABLE_DIR / "S2-04_matched_controls.csv.gz",
    "matching_quality": STEP2_TABLE_DIR / "S2-04_matching_quality.csv",
    "matching_audit": STEP2_TABLE_DIR / "S2-04_audit_summary.csv",
    "hourly_weather": DATA_DIR / "기상_시간단위_파생.csv",
}
missing = [str(p) for p in paths.values() if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required S3-01 input files:\n" + "\n".join(missing))

fire_unique = read_csv_kr(paths["fire_unique"], parse_dates=["기준시각"])
matching_quality = read_csv_kr(paths["matching_quality"], parse_dates=["기준시각", "date"])
matching_audit = read_csv_kr(paths["matching_audit"])
matched_controls = read_csv_kr(paths["matched_controls"], compression="gzip", parse_dates=["기준시각", "date", "fire_기준시각", "fire_date", "control_date"])

weather_header = read_csv_kr(paths["hourly_weather"], nrows=0).columns.tolist()
required_weather_cols = ["기상셀ID", "일시", "시점_기온_C", "시점_풍속_m_s", "시점_습도_pct"]
existing_rolling_cols = [spec[0] for spec in ROLLING_RECALC_SPECS if spec[0] in weather_header]
lag_cols = [c for c in ["D-1_최소습도_pct", "D-1_평균습도_pct", "D-1_강수량합_mm", "D-2_최소습도_pct", "D-3_최소습도_pct"] if c in weather_header]
weather_usecols = [c for c in dict.fromkeys(required_weather_cols + existing_rolling_cols + lag_cols) if c in weather_header]
hourly_weather = read_csv_kr(paths["hourly_weather"], usecols=weather_usecols, parse_dates=["일시"])
hourly_weather = hourly_weather.sort_values(["기상셀ID", "일시"]).reset_index(drop=True)

print("fire_unique", fire_unique.shape)
print("matching_quality", matching_quality.shape)
print("matched_controls", matched_controls.shape)
print("hourly_weather", hourly_weather.shape)
print("rolling cols", existing_rolling_cols)
print("lag cols", lag_cols)


In [4]:
# 산불 노출과 대조군을 같은 분석 키 구조로 정규화한다.
fire_base = matching_quality.copy()
fire_base["sample_group"] = "fire"
fire_base["target"] = 1
fire_base["control_rank"] = np.nan
fire_base["analysis_id"] = fire_base["fire_exposure_id"]
fire_base["기후지형유형"] = fire_base["climate_type"]

fire_cols = [
    "analysis_id", "fire_exposure_id", "control_rank", "sample_group", "target",
    "기상셀ID", "기준시각", "기후지형유형", "year", "month", "hour", "date",
    "raw_event_n", "fire_ids", "matched_control_n", "full_5_matched", "match_status"
]
fire_analysis = fire_base[fire_cols].copy()

control_base = matched_controls.copy()
control_base["sample_group"] = "matched_control"
control_base["target"] = 0
control_base["analysis_id"] = control_base["fire_exposure_id"].astype(str) + "_C" + control_base["control_rank"].astype(str).str.zfill(2)
if "기후지형유형" not in control_base.columns:
    control_base["기후지형유형"] = control_base.get("fire_climate_type", np.nan)
control_base = control_base.merge(
    matching_quality[["fire_exposure_id", "raw_event_n", "fire_ids", "matched_control_n", "full_5_matched", "match_status"]],
    on="fire_exposure_id",
    how="left",
    suffixes=("", "_fire")
)
control_cols = [
    "analysis_id", "fire_exposure_id", "control_rank", "sample_group", "target",
    "기상셀ID", "기준시각", "기후지형유형", "year", "month", "hour", "date",
    "raw_event_n", "fire_ids", "matched_control_n", "full_5_matched", "match_status"
]
control_analysis = control_base[control_cols].copy()

analysis_keys = pd.concat([fire_analysis, control_analysis], ignore_index=True)
analysis_keys["기준시각"] = pd.to_datetime(analysis_keys["기준시각"])
analysis_keys["date"] = pd.to_datetime(analysis_keys["date"]).dt.date.astype(str)
analysis_keys["year"] = analysis_keys["기준시각"].dt.year
analysis_keys["month"] = analysis_keys["기준시각"].dt.month
analysis_keys["hour"] = analysis_keys["기준시각"].dt.hour
analysis_keys["is_ys0071"] = analysis_keys["기상셀ID"].eq("YS_0071")
analysis_keys["is_less_than_5_matched"] = analysis_keys["matched_control_n"].fillna(0).lt(5)
analysis_keys["is_2021_feb_cluster"] = analysis_keys["기준시각"].dt.to_period("M").astype(str).eq("2021-02")
analysis_keys["is_february"] = analysis_keys["month"].eq(2)

# 기준시각에 해당하는 기존 rolling/lag 컬럼을 붙여 결측과 직접 재계산 비교에 쓴다.
point_cols = ["기상셀ID", "일시"] + existing_rolling_cols + lag_cols
point_weather = hourly_weather[point_cols].copy()
analysis_point = analysis_keys[["analysis_id", "기상셀ID", "기준시각", "sample_group", "기후지형유형"]].merge(
    point_weather,
    left_on=["기상셀ID", "기준시각"],
    right_on=["기상셀ID", "일시"],
    how="left"
)
analysis_keys["has_exact_hourly_row"] = analysis_point["일시"].notna().to_numpy()

# 시간창별 선행 원자료 관측 가능률: [기준시각-window, 기준시각)만 집계해 현재시각을 제외한다.
lead_rows = []
weather_groups = {cell: g for cell, g in hourly_weather.groupby("기상셀ID", sort=False)}
for cell, idx in analysis_keys.groupby("기상셀ID", sort=False).groups.items():
    g = weather_groups.get(cell)
    if g is None:
        for i in idx:
            for w in WINDOW_HOURS:
                lead_rows.append((analysis_keys.at[i, "analysis_id"], w, 0, False))
        continue
    times = g["일시"].to_numpy(dtype="datetime64[ns]")
    exp_times = analysis_keys.loc[idx, "기준시각"].to_numpy(dtype="datetime64[ns]")
    exp_ids = analysis_keys.loc[idx, "analysis_id"].to_numpy()
    for analysis_id, t in zip(exp_ids, exp_times):
        end = np.searchsorted(times, t, side="left")
        for w in WINDOW_HOURS:
            start = np.searchsorted(times, t - np.timedelta64(w, "h"), side="left")
            obs_n = int(end - start)
            lead_rows.append((analysis_id, w, obs_n, obs_n >= w))
lead_counts = pd.DataFrame(lead_rows, columns=["analysis_id", "window_h", "prior_obs_n", "available"])
lead_counts = lead_counts.merge(
    analysis_keys[["analysis_id", "sample_group", "target", "기상셀ID", "기준시각", "기후지형유형", "month", "hour", "fire_exposure_id"]],
    on="analysis_id",
    how="left"
)

# 기존 rolling 값이 현재시각 제외 방식과 일치하는지 직접 재계산한다.
recalc_rows = []
analysis_point_index = analysis_point.set_index("analysis_id")
for cell, idx in analysis_keys.groupby("기상셀ID", sort=False).groups.items():
    g = weather_groups.get(cell)
    if g is None:
        continue
    times = g["일시"].to_numpy(dtype="datetime64[ns]")
    for feature, source, window_h, agg in ROLLING_RECALC_SPECS:
        if feature not in hourly_weather.columns or source not in hourly_weather.columns:
            continue
        values = g[source].to_numpy(dtype="float64")
        for i in idx:
            analysis_id = analysis_keys.at[i, "analysis_id"]
            t = np.datetime64(analysis_keys.at[i, "기준시각"], "ns")
            end = np.searchsorted(times, t, side="left")
            start = np.searchsorted(times, t - np.timedelta64(window_h, "h"), side="left")
            window_values = values[start:end]
            window_values = window_values[~np.isnan(window_values)]
            recomputed = np.nan
            if len(window_values):
                if agg == "mean":
                    recomputed = float(np.mean(window_values))
                elif agg == "min":
                    recomputed = float(np.min(window_values))
                elif agg == "max":
                    recomputed = float(np.max(window_values))
            existing = analysis_point_index.at[analysis_id, feature] if analysis_id in analysis_point_index.index else np.nan
            abs_diff = np.nan if pd.isna(existing) or pd.isna(recomputed) else abs(float(existing) - recomputed)
            recalc_rows.append({
                "analysis_id": analysis_id,
                "sample_group": analysis_keys.at[i, "sample_group"],
                "기후지형유형": analysis_keys.at[i, "기후지형유형"],
                "feature": feature,
                "source_column": source,
                "window_h": window_h,
                "agg": agg,
                "prior_obs_n": int(end - start),
                "existing_value": existing,
                "recomputed_prior_only": recomputed,
                "abs_diff": abs_diff,
                "matches_prior_only": bool(pd.notna(abs_diff) and abs_diff <= 1e-9),
            })
recalc_detail = pd.DataFrame(recalc_rows)

# 표 요약 생성
input_rows = []
def add_input_row(item, value, expected=None, status=None, note=""):
    input_rows.append({"item": item, "value": value, "expected": expected, "status": status, "note": note})

add_input_row("S2 unique fire exposure rows", len(fire_unique), 1150, "ok" if len(fire_unique) == 1150 else "check")
add_input_row("S2 matching quality rows", len(matching_quality), 1150, "ok" if len(matching_quality) == 1150 else "check")
add_input_row("S2 matched control rows", len(matched_controls), 5632, "ok" if len(matched_controls) == 5632 else "check")
add_input_row("S3 analysis rows", len(analysis_keys), len(matching_quality) + len(matched_controls), "ok")
add_input_row("fire rows in S3", int((analysis_keys["target"] == 1).sum()), 1150, "ok" if int((analysis_keys["target"] == 1).sum()) == 1150 else "check")
add_input_row("control rows in S3", int((analysis_keys["target"] == 0).sum()), 5632, "ok" if int((analysis_keys["target"] == 0).sum()) == 5632 else "check")
add_input_row("full 5 matched fire exposures", int(matching_quality["full_5_matched"].sum()), 1110, "ok" if int(matching_quality["full_5_matched"].sum()) == 1110 else "check")
add_input_row("less than 5 matched fire exposures", int((~matching_quality["full_5_matched"].astype(bool)).sum()), 40, "ok" if int((~matching_quality["full_5_matched"].astype(bool)).sum()) == 40 else "check")
add_input_row("analysis key duplicate rows", int(analysis_keys.duplicated(["analysis_id"]).sum()), 0, "ok" if not analysis_keys.duplicated(["analysis_id"]).any() else "fail")
add_input_row("cell-time duplicate within sample_group", int(analysis_keys.duplicated(["sample_group", "기상셀ID", "기준시각", "fire_exposure_id", "control_rank"]).sum()), 0, "ok")
add_input_row("exact hourly row coverage pct", round(float(analysis_keys["has_exact_hourly_row"].mean() * 100), 4), 100, "ok" if analysis_keys["has_exact_hourly_row"].all() else "check")
add_input_row("S2-04 matched_controls sha256", sha256_file(paths["matched_controls"]), None, "info")
for _, row in matching_audit.iterrows():
    add_input_row(f"S2 audit: {row.get('metric') or row.iloc[0]}", row.get("value", row.iloc[-1]), None, "info")
input_audit = pd.DataFrame(input_rows)

lead_summary = (
    lead_counts.groupby(["sample_group", "기후지형유형", "window_h"], dropna=False)
    .agg(
        exposure_n=("analysis_id", "nunique"),
        available_n=("available", "sum"),
        availability_pct=("available", lambda s: float(s.mean() * 100)),
        mean_prior_obs_n=("prior_obs_n", "mean"),
        min_prior_obs_n=("prior_obs_n", "min"),
    )
    .reset_index()
)
existing_feature_rows = []
for feature in existing_rolling_cols + lag_cols:
    tmp = analysis_point[["analysis_id", "sample_group", "기후지형유형", feature]].copy()
    tmp["available"] = tmp[feature].notna()
    g = tmp.groupby(["sample_group", "기후지형유형"], dropna=False)["available"].agg(["count", "sum", "mean"]).reset_index()
    g["feature"] = feature
    g["window_h"] = np.nan
    g = g.rename(columns={"count": "exposure_n", "sum": "available_n", "mean": "availability_pct"})
    g["availability_pct"] = g["availability_pct"] * 100
    existing_feature_rows.append(g[["sample_group", "기후지형유형", "feature", "window_h", "exposure_n", "available_n", "availability_pct"]])
existing_feature_summary = pd.concat(existing_feature_rows, ignore_index=True) if existing_feature_rows else pd.DataFrame()
lead_feature_summary = lead_summary.copy()
lead_feature_summary["feature"] = "prior_hourly_obs_" + lead_feature_summary["window_h"].astype(int).astype(str) + "h"
lead_feature_summary = lead_feature_summary[["sample_group", "기후지형유형", "feature", "window_h", "exposure_n", "available_n", "availability_pct", "mean_prior_obs_n", "min_prior_obs_n"]]
lead_feature_availability = pd.concat([lead_feature_summary, existing_feature_summary], ignore_index=True, sort=False)

if recalc_detail.empty:
    leakage_recalc_audit = pd.DataFrame(columns=["feature", "sample_group", "n", "comparable_n", "match_n", "mismatch_n", "max_abs_diff", "status"])
else:
    leakage_recalc_audit = (
        recalc_detail.groupby(["feature", "sample_group"], dropna=False)
        .agg(
            n=("analysis_id", "count"),
            comparable_n=("abs_diff", lambda s: int(s.notna().sum())),
            match_n=("matches_prior_only", "sum"),
            mismatch_n=("matches_prior_only", lambda s: int((~s.astype(bool)).sum())),
            max_abs_diff=("abs_diff", "max"),
            median_abs_diff=("abs_diff", "median"),
            min_prior_obs_n=("prior_obs_n", "min"),
        )
        .reset_index()
    )
    leakage_recalc_audit["status"] = np.where(
        (leakage_recalc_audit["comparable_n"] > 0) & (leakage_recalc_audit["max_abs_diff"].fillna(0) <= 1e-9),
        "ok_prior_only_matches_existing",
        "check_mismatch_or_missing"
    )

flagged_exposures = analysis_keys[[
    "analysis_id", "fire_exposure_id", "control_rank", "sample_group", "target", "기상셀ID", "기준시각", "기후지형유형",
    "month", "hour", "date", "matched_control_n", "full_5_matched", "is_less_than_5_matched", "is_february", "is_2021_feb_cluster", "is_ys0071", "has_exact_hourly_row"
]].copy()

# 저장
written_tables = []
for df, name in [
    (input_audit, "S3-01_input_population_audit.csv"),
    (lead_feature_availability, "S3-01_lead_feature_availability.csv"),
    (leakage_recalc_audit, "S3-01_leakage_recalculation_audit.csv"),
    (recalc_detail, "S3-01_leakage_recalculation_detail.csv"),
    (flagged_exposures, "S3-01_flagged_exposures.csv"),
]:
    written_tables.append(write_table(df, name))

input_audit


In [5]:
# S3-01 플롯: 선행 원자료 관측 가능률과 기존 rolling/lag 결측을 한눈에 점검한다.
plot_df = lead_feature_availability.copy()
plot_df["feature_label"] = np.where(
    plot_df["window_h"].notna(),
    "prior_obs_" + plot_df["window_h"].fillna(0).astype(int).astype(str) + "h",
    plot_df["feature"].astype(str)
)
plot_df["group_label"] = plot_df["sample_group"].astype(str) + " | " + plot_df["기후지형유형"].astype(str)
pivot = plot_df.pivot_table(index="feature_label", columns="group_label", values="availability_pct", aggfunc="mean")

height = max(4, 0.34 * len(pivot.index) + 1.5)
width = max(8, 0.7 * len(pivot.columns) + 2)
plt.figure(figsize=(width, height))
sns.heatmap(pivot, vmin=0, vmax=100, cmap="viridis", annot=True, fmt=".1f", linewidths=0.4, linecolor="white", cbar_kws={"label": "availability (%)"})
plt.title("S3-01 lead feature availability by sample group and climate type")
plt.xlabel("")
plt.ylabel("")
plt.xticks(rotation=35, ha="right")
heatmap_path = save_current_plot("S3-01_feature_availability_heatmap.png")

missing_position = lead_counts.copy()
missing_position["missing"] = ~missing_position["available"]
missing_summary = (
    missing_position.groupby(["기상셀ID", "window_h"], dropna=False)
    .agg(total_n=("analysis_id", "count"), missing_n=("missing", "sum"), missing_pct=("missing", lambda s: float(s.mean() * 100)))
    .reset_index()
)
missing_pivot = missing_summary.pivot(index="기상셀ID", columns="window_h", values="missing_pct").fillna(0)
plt.figure(figsize=(7, max(5, 0.12 * len(missing_pivot.index))))
sns.heatmap(missing_pivot, vmin=0, vmax=100, cmap="mako_r", cbar_kws={"label": "missing (%)"})
plt.title("S3-01 missing prior-hour coverage by weather cell")
plt.xlabel("window (h)")
plt.ylabel("weather cell")
missing_plot_path = save_current_plot("S3-01_missing_prior_hour_coverage_by_cell.png")

written_plots = [heatmap_path, missing_plot_path]
artifact_manifest = pd.DataFrame([
    {"artifact_type": "table", "path": str(p.relative_to(REPO_ROOT)), "rows": int(pd.read_csv(p, encoding="utf-8-sig").shape[0])}
    for p in written_tables
] + [
    {"artifact_type": "plot", "path": str(p.relative_to(REPO_ROOT)), "rows": np.nan}
    for p in written_plots
])
manifest_path = write_table(artifact_manifest, "S3-01_artifact_manifest.csv")

print("written tables")
for p in written_tables + [manifest_path]:
    print(" -", p)
print("written plots")
for p in written_plots:
    print(" -", p)

print("\nInput audit status counts")
print(input_audit["status"].value_counts(dropna=False))
print("\nLead availability minimum by window")
print(lead_feature_availability[lead_feature_availability["window_h"].notna()].groupby("window_h")["availability_pct"].min())
print("\nLeakage recalculation status")
print(leakage_recalc_audit["status"].value_counts(dropna=False) if not leakage_recalc_audit.empty else "no comparable rolling columns")


## S3-02 : 분석 범위 고정

분석 질문, 단위, 모집단, 비교집단, 기간, 통제·층화·매칭 조건을 고정한다.


In [7]:
S3_ID = "S3-02"
S3_02_SCOPE = pd.DataFrame([
    {"item": "analysis_question", "value": "1~72h 선행 습도·강수·풍속과 as-of FFMC/ISI/FWI 중 매칭 대조군 대비 delta가 가장 안정적인 시간창을 찾는다."},
    {"item": "analysis_unit", "value": "fire_exposure_id별 대응 비교: 산불 선행값 - 동일 fire_exposure_id 매칭 대조군 평균"},
    {"item": "population", "value": "Step2 고유 산불 노출 1,150개"},
    {"item": "comparison_group", "value": "Step2 고정 매칭 대조군 5,632행"},
    {"item": "period", "value": "2020-01-01 00:00부터 2021-12-31 23:00까지의 강원도 기상셀 시간자료와 캐나다 일단위 as-of 지수"},
    {"item": "matching_control", "value": "동일 기상셀ID, 동일 연도, 동일 월, 동일 hour. 산불 동일 노출 및 동일 셀 전후 48시간 제외는 Step2 매칭 결과를 그대로 사용."},
    {"item": "stratification", "value": "전체, 영동 해안형, 영서 내륙형, 고지·산간형"},
    {"item": "leakage_rule", "value": "시간기상 파생은 [기준시각-window, 기준시각) 반개구간으로 현재시각을 제외. 캐나다 지수는 기준시각 이전 또는 같은 정오 as-of만 사용."},
])
scope_path = write_table(S3_02_SCOPE, "S3-02_analysis_scope.csv")
S3_02_SCOPE


## S3-02 : 선행창 파생 및 입력 감사

선행 1/3/6/12/24/48/72시간 기상값과 캐나다 as-of 지수를 산불·대조군에 동일 함수로 결합하고 입력 감사를 저장한다.


In [9]:
from scipy import stats

raw_weather_path = REPO_ROOT / "data" / "강원도_날씨데이터" / "강원도날씨_격자_시간단위.csv"
fwi_path = DATA_DIR / "캐나다_FWI_일단위.csv"
if not raw_weather_path.exists():
    raise FileNotFoundError(raw_weather_path)
if not fwi_path.exists():
    raise FileNotFoundError(fwi_path)

raw_weather = read_csv_kr(
    raw_weather_path,
    usecols=["기상셀ID", "일시", "습도_pct", "풍속_m_s", "강수량_mm"],
    parse_dates=["일시"],
).sort_values(["기상셀ID", "일시"]).reset_index(drop=True)
fwi_daily = read_csv_kr(fwi_path, parse_dates=["날짜"])

if "analysis_keys" not in globals():
    raise RuntimeError("S3-01 analysis_keys가 필요합니다. 노트북을 처음부터 실행하세요.")
if analysis_keys["analysis_id"].duplicated().any():
    raise ValueError("analysis_id duplicated before S3-02")

s3_input = analysis_keys.copy()
s3_input["기준시각"] = pd.to_datetime(s3_input["기준시각"])
s3_input["date"] = s3_input["기준시각"].dt.date.astype(str)
s3_input["canadian_ref_date"] = (s3_input["기준시각"] - pd.to_timedelta((s3_input["기준시각"].dt.hour < 12).astype(int), unit="D")).dt.normalize()
s3_input["canadian_ref_time"] = s3_input["canadian_ref_date"] + pd.Timedelta(hours=12)
s3_input["canadian_lag_hours"] = (s3_input["기준시각"] - s3_input["canadian_ref_time"]).dt.total_seconds() / 3600
if (s3_input["canadian_ref_time"] > s3_input["기준시각"]).any():
    raise ValueError("Canadian as-of time leakage detected")

feature_rows = []
weather_groups = {cell: g for cell, g in raw_weather.groupby("기상셀ID", sort=False)}
source_specs = [
    ("humidity_mean", "습도_pct", "mean", "lower"),
    ("humidity_min", "습도_pct", "min", "lower"),
    ("wind_mean", "풍속_m_s", "mean", "higher"),
    ("wind_max", "풍속_m_s", "max", "higher"),
    ("rain_sum", "강수량_mm", "sum", "lower"),
]
feature_meta_rows = []
for variable, source, agg, risk_direction in source_specs:
    for window_h in WINDOW_HOURS:
        feature_meta_rows.append({"feature": f"{variable}_{window_h}h", "variable": variable, "source_column": source, "window_h": window_h, "agg": agg, "risk_direction": risk_direction, "feature_family": "hourly_window"})

for cell, idx in s3_input.groupby("기상셀ID", sort=False).groups.items():
    g = weather_groups.get(cell)
    if g is None:
        for i in idx:
            feature_rows.append({"analysis_id": s3_input.at[i, "analysis_id"]})
        continue
    times = g["일시"].to_numpy(dtype="datetime64[ns]")
    arrays = {col: g[col].to_numpy(dtype="float64") for col in ["습도_pct", "풍속_m_s", "강수량_mm"]}
    for i in idx:
        t = np.datetime64(s3_input.at[i, "기준시각"], "ns")
        end = np.searchsorted(times, t, side="left")
        rec = {"analysis_id": s3_input.at[i, "analysis_id"]}
        for variable, source, agg, risk_direction in source_specs:
            vals = arrays[source]
            for window_h in WINDOW_HOURS:
                start = np.searchsorted(times, t - np.timedelta64(window_h, "h"), side="left")
                win = vals[start:end]
                win = win[~np.isnan(win)]
                out = np.nan
                if len(win) >= window_h:
                    if agg == "mean": out = float(np.mean(win))
                    elif agg == "min": out = float(np.min(win))
                    elif agg == "max": out = float(np.max(win))
                    elif agg == "sum": out = float(np.sum(win))
                rec[f"{variable}_{window_h}h"] = out
        feature_rows.append(rec)

lead_features = pd.DataFrame(feature_rows)
features_wide = s3_input.merge(lead_features, on="analysis_id", how="left")
fwi_merge = fwi_daily[["기상셀ID", "날짜", "FFMC", "ISI", "FWI"]].rename(columns={"날짜": "canadian_ref_date"})
features_wide = features_wide.merge(fwi_merge, on=["기상셀ID", "canadian_ref_date"], how="left", validate="many_to_one")
for v in ["FFMC", "ISI", "FWI"]:
    feature_meta_rows.append({"feature": v, "variable": v, "source_column": "캐나다_FWI_일단위.csv", "window_h": np.nan, "agg": "asof_no_future", "risk_direction": "higher", "feature_family": "canadian_asof"})
feature_meta = pd.DataFrame(feature_meta_rows)

feature_list = feature_meta["feature"].tolist()
missingness = []
for feature in feature_list:
    for group_name, sub in features_wide.groupby("sample_group", dropna=False):
        missingness.append({"feature": feature, "sample_group": group_name, "row_n": len(sub), "missing_n": int(sub[feature].isna().sum()), "missing_pct": float(sub[feature].isna().mean() * 100)})
missingness_audit = pd.DataFrame(missingness).merge(feature_meta, on="feature", how="left")

fire_wide = features_wide[features_wide["target"].eq(1)].copy()
control_wide = features_wide[features_wide["target"].eq(0)].copy()
control_means = control_wide.groupby("fire_exposure_id", as_index=False)[feature_list].mean(numeric_only=True)
control_counts = control_wide.groupby("fire_exposure_id", as_index=False).agg(control_row_n=("analysis_id", "count"))
paired = fire_wide.merge(control_means, on="fire_exposure_id", how="left", suffixes=("_fire", "_control_mean"))
paired = paired.merge(control_counts, on="fire_exposure_id", how="left")

raw_long_rows = []
for _, meta in feature_meta.iterrows():
    feature = meta["feature"]
    fcol = f"{feature}_fire"
    ccol = f"{feature}_control_mean"
    tmp = paired[["fire_exposure_id", "기상셀ID", "기준시각", "기후지형유형", "date", "month", "hour", "matched_control_n", "full_5_matched", "is_ys0071", "is_less_than_5_matched", "is_2021_feb_cluster", "control_row_n", fcol, ccol]].copy()
    tmp = tmp.rename(columns={fcol: "fire_value", ccol: "control_mean"})
    tmp["feature"] = feature
    tmp["variable"] = meta["variable"]
    tmp["window_h"] = meta["window_h"]
    tmp["risk_direction"] = meta["risk_direction"]
    tmp["feature_family"] = meta["feature_family"]
    tmp["delta"] = tmp["fire_value"] - tmp["control_mean"]
    tmp["risk_direction_match"] = np.where(tmp["risk_direction"].eq("lower"), tmp["delta"] < 0, tmp["delta"] > 0)
    raw_long_rows.append(tmp)
delta_raw = pd.concat(raw_long_rows, ignore_index=True)

population_audit = pd.DataFrame([
    {"item": "analysis_unit", "value": "fire_exposure_id paired delta"},
    {"item": "fire_exposure_n", "value": int(fire_wide["fire_exposure_id"].nunique())},
    {"item": "fire_row_n", "value": int(len(fire_wide))},
    {"item": "control_row_n", "value": int(len(control_wide))},
    {"item": "paired_fire_exposure_n", "value": int(paired["fire_exposure_id"].nunique())},
    {"item": "delta_raw_row_n", "value": int(len(delta_raw))},
    {"item": "analysis_id_duplicate_n", "value": int(features_wide["analysis_id"].duplicated().sum())},
    {"item": "fire_exposure_duplicate_in_fire_n", "value": int(fire_wide["fire_exposure_id"].duplicated().sum())},
    {"item": "control_without_fire_id_n", "value": int(control_wide["fire_exposure_id"].isna().sum())},
    {"item": "canadian_asof_future_violation_n", "value": int((s3_input["canadian_ref_time"] > s3_input["기준시각"]).sum())},
])
write_table(population_audit, "S3-02_population_audit.csv")
write_table(missingness_audit, "S3-02_feature_missingness_audit.csv")
write_table(delta_raw, "S3-02_lead_window_delta_raw.csv")
write_table(feature_meta, "S3-02_feature_registry.csv")
print(population_audit)
print(missingness_audit.groupby("sample_group")["missing_pct"].max())


## S3-02 : 대응 delta 통계 요약

`fire_exposure_id`별 산불값 minus 매칭 대조군 평균 delta를 요약하고 date/cell block bootstrap, p/q값, 효과크기를 계산한다.


In [11]:
BOOT_N = 500
rng = np.random.default_rng(42)

def bh_fdr(p_values):
    p = np.asarray(p_values, dtype=float)
    q = np.full_like(p, np.nan, dtype=float)
    valid = np.isfinite(p)
    if valid.sum() == 0: return q
    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    m = len(ranked)
    adj = ranked * m / np.arange(1, m + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0, 1)
    valid_idx = np.where(valid)[0]
    q[valid_idx[order]] = adj
    return q

def bootstrap_ci(values, blocks, reps=BOOT_N):
    vals = np.asarray(values, dtype=float)
    block_arr = np.asarray(blocks)
    ok = np.isfinite(vals) & pd.notna(block_arr)
    vals = vals[ok]
    block_arr = block_arr[ok]
    if len(vals) < 2: return np.nan, np.nan
    unique_blocks = pd.unique(block_arr)
    if len(unique_blocks) < 2: return np.nan, np.nan
    by_block = {b: vals[block_arr == b] for b in unique_blocks}
    means = np.empty(reps)
    for r in range(reps):
        sample_blocks = rng.choice(unique_blocks, size=len(unique_blocks), replace=True)
        sample_vals = np.concatenate([by_block[b] for b in sample_blocks])
        means[r] = np.nanmean(sample_vals)
    return float(np.nanpercentile(means, 2.5)), float(np.nanpercentile(means, 97.5))

def summarize_delta(sub, group_label):
    rows = []
    meta_lookup = feature_meta.set_index("feature")
    for feature, g in sub.groupby("feature", sort=False):
        meta = meta_lookup.loc[feature]
        complete = g[g["delta"].notna()].copy()
        x = complete["delta"].to_numpy(dtype=float)
        n = len(x)
        if n == 0: continue
        mean_delta = float(np.mean(x))
        sd_delta = float(np.std(x, ddof=1)) if n > 1 else np.nan
        dz = mean_delta / sd_delta if sd_delta and np.isfinite(sd_delta) and sd_delta != 0 else np.nan
        se = sd_delta / math.sqrt(n) if n > 1 and np.isfinite(sd_delta) else np.nan
        ci_low = mean_delta - stats.t.ppf(0.975, n - 1) * se if n > 1 and np.isfinite(se) else np.nan
        ci_high = mean_delta + stats.t.ppf(0.975, n - 1) * se if n > 1 and np.isfinite(se) else np.nan
        try: t_p = float(stats.ttest_1samp(x, 0, nan_policy="omit").pvalue) if n > 1 else np.nan
        except Exception: t_p = np.nan
        try:
            nonzero = x[x != 0]
            w_p = float(stats.wilcoxon(nonzero).pvalue) if len(nonzero) > 0 else np.nan
        except Exception: w_p = np.nan
        date_ci_low, date_ci_high = bootstrap_ci(x, complete["date"].to_numpy())
        cell_ci_low, cell_ci_high = bootstrap_ci(x, complete["기상셀ID"].to_numpy())
        risk_direction = meta["risk_direction"]
        direction_pct = float(((x < 0) if risk_direction == "lower" else (x > 0)).mean() * 100)
        rows.append({"group": group_label, "feature": feature, "variable": meta["variable"], "feature_family": meta["feature_family"], "window_h": meta["window_h"], "risk_direction": risk_direction, "n": int(n), "fire_mean": float(complete["fire_value"].mean()), "control_mean": float(complete["control_mean"].mean()), "mean_delta": mean_delta, "median_delta": float(np.median(x)), "sd_delta": sd_delta, "mean_delta_ci_low_t": ci_low, "mean_delta_ci_high_t": ci_high, "date_boot_ci_low": date_ci_low, "date_boot_ci_high": date_ci_high, "cell_boot_ci_low": cell_ci_low, "cell_boot_ci_high": cell_ci_high, "cohen_dz": dz, "direction_match_pct": direction_pct, "paired_t_p": t_p, "wilcoxon_p": w_p, "date_n": int(complete["date"].nunique()), "cell_n": int(complete["기상셀ID"].nunique()), "support_flag": "ok" if (n >= 30 and complete["date"].nunique() >= 10 and complete["기상셀ID"].nunique() >= 5) else "sparse_do_not_interpret"})
    return pd.DataFrame(rows)

summary_parts = [summarize_delta(delta_raw, "전체")]
for climate_type, sub in delta_raw.groupby("기후지형유형", sort=False):
    summary_parts.append(summarize_delta(sub, climate_type))
delta_summary = pd.concat(summary_parts, ignore_index=True)
delta_summary["paired_t_q"] = bh_fdr(delta_summary["paired_t_p"])
delta_summary["wilcoxon_q"] = bh_fdr(delta_summary["wilcoxon_p"])
delta_summary["date_ci_excludes_zero"] = (delta_summary["date_boot_ci_low"] > 0) | (delta_summary["date_boot_ci_high"] < 0)
delta_summary["cell_ci_excludes_zero"] = (delta_summary["cell_boot_ci_low"] > 0) | (delta_summary["cell_boot_ci_high"] < 0)
delta_summary["candidate_priority"] = np.select([delta_summary["support_flag"].ne("ok"), delta_summary["date_ci_excludes_zero"] & delta_summary["cell_ci_excludes_zero"] & delta_summary["direction_match_pct"].ge(60), delta_summary["date_ci_excludes_zero"] & delta_summary["direction_match_pct"].ge(55)], ["drop_sparse", "strong_window_candidate", "weak_window_candidate"], default="descriptive_only")
by_climate = delta_summary[delta_summary["group"].ne("전체")].copy()
rank_overall = delta_summary[delta_summary["group"].eq("전체")].copy()
rank_overall["abs_cohen_dz"] = rank_overall["cohen_dz"].abs()
rank_overall = rank_overall.sort_values("abs_cohen_dz", ascending=False)
write_table(delta_summary, "S3-02_lead_window_delta_summary.csv")
write_table(by_climate, "S3-02_lead_window_by_climate_type.csv")
write_table(rank_overall, "S3-02_window_rank_overall.csv")
print(delta_summary[delta_summary["group"].eq("전체")].sort_values("cohen_dz", key=lambda s: s.abs(), ascending=False).head(12)[["feature", "n", "mean_delta", "date_boot_ci_low", "date_boot_ci_high", "cell_boot_ci_low", "cell_boot_ci_high", "cohen_dz", "direction_match_pct", "paired_t_p", "paired_t_q", "candidate_priority"]])


## S3-02 : 시간창별 효과 플롯 저장

시간창별 효과크기 플롯과 전체 순위 forest plot을 저장한다.


In [13]:
plot_paths = []
plot_groups = ["전체", "영동 해안형", "영서 내륙형", "고지·산간형"]
plot_palette = {"전체": "#222222", "영동 해안형": "#1b9e77", "영서 내륙형": "#377eb8", "고지·산간형": "#d95f02"}
plot_labels = {"humidity_min": "Minimum humidity delta (%p)", "humidity_mean": "Mean humidity delta (%p)", "wind_mean": "Mean wind speed delta (m/s)", "wind_max": "Max wind speed delta (m/s)", "rain_sum": "Rainfall sum delta (mm)"}
for variable in ["humidity_min", "humidity_mean", "wind_mean", "wind_max", "rain_sum"]:
    pdat = delta_summary[(delta_summary["variable"].eq(variable)) & (delta_summary["group"].isin(plot_groups))].copy()
    pdat = pdat[pdat["window_h"].notna()].sort_values(["group", "window_h"])
    plt.figure(figsize=(8.5, 5.0))
    for group in plot_groups:
        sub = pdat[pdat["group"].eq(group)]
        if sub.empty: continue
        plt.plot(sub["window_h"], sub["mean_delta"], marker="o", linewidth=2, label=group, color=plot_palette[group])
        if group == "전체":
            x = sub["window_h"].astype(float).to_numpy()
            lo = sub["date_boot_ci_low"].astype(float).to_numpy()
            hi = sub["date_boot_ci_high"].astype(float).to_numpy()
            plt.fill_between(x, lo, hi, color=plot_palette[group], alpha=0.15)
    plt.axhline(0, color="#666666", linewidth=1)
    plt.xticks(WINDOW_HOURS)
    plt.xlabel("Prior window (hours, current hour excluded)")
    plt.ylabel(plot_labels[variable])
    plt.title(f"S3-02 {variable}: paired fire-control delta by lead window")
    plt.legend(frameon=True)
    plot_paths.append(save_current_plot(f"S3-02_effect_by_window_{variable}.png"))

can = delta_summary[(delta_summary["feature_family"].eq("canadian_asof")) & (delta_summary["group"].isin(plot_groups))].copy()
plt.figure(figsize=(8.0, 4.8))
sns.pointplot(data=can, x="feature", y="mean_delta", hue="group", order=["FFMC", "ISI", "FWI"], hue_order=plot_groups, dodge=0.35, errorbar=None, palette=plot_palette)
for i, feature in enumerate(["FFMC", "ISI", "FWI"]):
    sub = can[(can["feature"].eq(feature)) & (can["group"].eq("전체"))]
    if not sub.empty:
        row = sub.iloc[0]
        plt.errorbar(i - 0.27, row["mean_delta"], yerr=[[row["mean_delta"] - row["date_boot_ci_low"]], [row["date_boot_ci_high"] - row["mean_delta"]]], fmt="none", color="#222222", capsize=4, linewidth=1.5)
plt.axhline(0, color="#666666", linewidth=1)
plt.xlabel("")
plt.ylabel("As-of index delta")
plt.title("S3-02 Canadian indices: paired fire-control delta")
plt.legend(frameon=True, title="")
plot_paths.append(save_current_plot("S3-02_effect_asof_canadian_indices.png"))

forest = rank_overall.head(18).copy().sort_values("cohen_dz")
forest["label"] = forest.apply(lambda r: f"{r['feature']}" if pd.isna(r["window_h"]) else f"{r['variable']} {int(r['window_h'])}h", axis=1)
plt.figure(figsize=(8.5, max(5, 0.34 * len(forest) + 1.6)))
y = np.arange(len(forest))
scaled_low = (forest["mean_delta"] - forest["date_boot_ci_low"]).abs() / forest["sd_delta"].replace(0, np.nan)
scaled_high = (forest["date_boot_ci_high"] - forest["mean_delta"]).abs() / forest["sd_delta"].replace(0, np.nan)
plt.errorbar(forest["cohen_dz"], y, xerr=[scaled_low, scaled_high], fmt="o", color="#333333", ecolor="#888888", capsize=3)
plt.axvline(0, color="#666666", linewidth=1)
plt.yticks(y, forest["label"])
plt.xlabel("Cohen's dz (date-bootstrap CI scaled by delta SD)")
plt.ylabel("")
plt.title("S3-02 strongest overall paired deltas")
plot_paths.append(save_current_plot("S3-02_window_rank_forest_plot.png"))

plot_manifest = pd.DataFrame([{"plot_path": str(p.relative_to(REPO_ROOT)), "file_name": p.name} for p in plot_paths])
write_table(plot_manifest, "S3-02_plot_manifest.csv")
artifact_manifest = pd.DataFrame([{ "artifact_type": "table", "path": str((TABLE_DIR / name).relative_to(REPO_ROOT)) } for name in ["S3-02_analysis_scope.csv", "S3-02_population_audit.csv", "S3-02_feature_missingness_audit.csv", "S3-02_feature_registry.csv", "S3-02_lead_window_delta_raw.csv", "S3-02_lead_window_delta_summary.csv", "S3-02_lead_window_by_climate_type.csv", "S3-02_window_rank_overall.csv", "S3-02_plot_manifest.csv"]] + [{"artifact_type": "plot", "path": str(p.relative_to(REPO_ROOT))} for p in plot_paths])
write_table(artifact_manifest, "S3-02_artifact_manifest.csv")
print(plot_manifest)


## S3-03 : 분석 범위 고정

급건조·급변 프록시의 분석 질문, 단위, 모집단, 비교집단, 기간, 통제·층화·매칭 조건을 고정한다.


In [15]:
S3_ID = "S3-03"
S3_03_SCOPE = pd.DataFrame([
    {"item": "analysis_question", "value": "발생 전 습도 급락, 풍속 증가, 기압 급변, FFMC/ISI 상승이 동일 셀·연도·월·hour 매칭 대조군 대비 두드러지는가?"},
    {"item": "analysis_unit", "value": "fire_exposure_id별 대응 비교: 산불 급변 프록시값 - 동일 fire_exposure_id 매칭 대조군 평균"},
    {"item": "population", "value": "Step2 고유 산불 노출 1,150개"},
    {"item": "comparison_group", "value": "Step2 고정 매칭 대조군 5,632행"},
    {"item": "period", "value": "2020-01-01 00:00부터 2021-12-31 23:00까지의 강원도 기상셀 시간자료와 캐나다 일단위 as-of 지수"},
    {"item": "matching_control", "value": "동일 기상셀ID, 동일 연도, 동일 월, 동일 hour. 산불 동일 노출 및 동일 셀 전후 48시간 제외는 Step2 매칭 결과를 그대로 사용."},
    {"item": "stratification", "value": "전체, 영동 해안형, 영서 내륙형, 고지·산간형"},
    {"item": "leakage_rule", "value": "모든 변화량은 기준시각 현재를 제외한 최근창과 그 이전창의 차이로 계산한다. 캐나다 지수 변화량은 기준시각 이전 as-of 날짜와 그 전날만 사용한다."},
])
write_table(S3_03_SCOPE, "S3-03_analysis_scope.csv")
S3_03_SCOPE


## S3-03 : 급변 프록시 파생 및 입력 감사

기준시각 현재를 제외한 최근창과 이전창의 차이로 습도·풍속·기압·캐나다 지수 변화량을 산불과 대조군에 동일하게 계산한다.


In [17]:
from scipy import stats

raw_weather_path = REPO_ROOT / "data" / "강원도_날씨데이터" / "강원도날씨_격자_시간단위.csv"
fwi_path = DATA_DIR / "캐나다_FWI_일단위.csv"
if not raw_weather_path.exists():
    raise FileNotFoundError(raw_weather_path)
if not fwi_path.exists():
    raise FileNotFoundError(fwi_path)

raw_weather_s303 = read_csv_kr(
    raw_weather_path,
    usecols=["기상셀ID", "일시", "습도_pct", "풍속_m_s", "현지기압_hPa"],
    parse_dates=["일시"],
).sort_values(["기상셀ID", "일시"]).reset_index(drop=True)
fwi_daily_s303 = read_csv_kr(fwi_path, parse_dates=["날짜"])

if "analysis_keys" not in globals():
    raise RuntimeError("S3-01 analysis_keys가 필요합니다. 노트북을 처음부터 실행하세요.")
if analysis_keys["analysis_id"].duplicated().any():
    raise ValueError("analysis_id duplicated before S3-03")

s3_03_input = analysis_keys.copy()
s3_03_input["기준시각"] = pd.to_datetime(s3_03_input["기준시각"])
s3_03_input["date"] = s3_03_input["기준시각"].dt.date.astype(str)
s3_03_input["canadian_ref_date"] = (s3_03_input["기준시각"] - pd.to_timedelta((s3_03_input["기준시각"].dt.hour < 12).astype(int), unit="D")).dt.normalize()
s3_03_input["canadian_ref_time"] = s3_03_input["canadian_ref_date"] + pd.Timedelta(hours=12)
s3_03_input["canadian_prev_date"] = s3_03_input["canadian_ref_date"] - pd.Timedelta(days=1)
if (s3_03_input["canadian_ref_time"] > s3_03_input["기준시각"]).any():
    raise ValueError("Canadian as-of time leakage detected in S3-03")

rapid_defs = [
    {"feature": "rh_mean_1h_minus_prev6h_mean", "variable": "humidity_drop", "source": "습도_pct", "recent_start_h": 0, "recent_end_h": 1, "recent_agg": "mean", "prev_start_h": 1, "prev_end_h": 7, "prev_agg": "mean", "risk_direction": "lower", "unit": "%p"},
    {"feature": "rh_mean_3h_minus_prev21h_mean", "variable": "humidity_drop", "source": "습도_pct", "recent_start_h": 0, "recent_end_h": 3, "recent_agg": "mean", "prev_start_h": 3, "prev_end_h": 24, "prev_agg": "mean", "risk_direction": "lower", "unit": "%p"},
    {"feature": "rh_mean_6h_minus_prev18h_mean", "variable": "humidity_drop", "source": "습도_pct", "recent_start_h": 0, "recent_end_h": 6, "recent_agg": "mean", "prev_start_h": 6, "prev_end_h": 24, "prev_agg": "mean", "risk_direction": "lower", "unit": "%p"},
    {"feature": "rh_min_6h_minus_prev18h_mean", "variable": "humidity_drop", "source": "습도_pct", "recent_start_h": 0, "recent_end_h": 6, "recent_agg": "min", "prev_start_h": 6, "prev_end_h": 24, "prev_agg": "mean", "risk_direction": "lower", "unit": "%p"},
    {"feature": "wind_mean_3h_minus_prev21h_mean", "variable": "wind_increase", "source": "풍속_m_s", "recent_start_h": 0, "recent_end_h": 3, "recent_agg": "mean", "prev_start_h": 3, "prev_end_h": 24, "prev_agg": "mean", "risk_direction": "higher", "unit": "m/s"},
    {"feature": "wind_max_6h_minus_prev18h_max", "variable": "wind_increase", "source": "풍속_m_s", "recent_start_h": 0, "recent_end_h": 6, "recent_agg": "max", "prev_start_h": 6, "prev_end_h": 24, "prev_agg": "max", "risk_direction": "higher", "unit": "m/s"},
    {"feature": "pressure_mean_3h_minus_prev3h_mean", "variable": "pressure_change", "source": "현지기압_hPa", "recent_start_h": 0, "recent_end_h": 3, "recent_agg": "mean", "prev_start_h": 3, "prev_end_h": 6, "prev_agg": "mean", "risk_direction": "two_sided", "unit": "hPa"},
]
rapid_meta = pd.DataFrame(rapid_defs)

def agg_interval(values, times, t, start_h, end_h, agg):
    start = np.searchsorted(times, t - np.timedelta64(end_h, "h"), side="left")
    end = np.searchsorted(times, t - np.timedelta64(start_h, "h"), side="left")
    win = values[start:end]
    win = win[~np.isnan(win)]
    if len(win) < (end_h - start_h):
        return np.nan
    if agg == "mean": return float(np.mean(win))
    if agg == "min": return float(np.min(win))
    if agg == "max": return float(np.max(win))
    raise ValueError(agg)

feature_rows = []
weather_groups_s303 = {cell: g for cell, g in raw_weather_s303.groupby("기상셀ID", sort=False)}
for cell, idx in s3_03_input.groupby("기상셀ID", sort=False).groups.items():
    g = weather_groups_s303.get(cell)
    if g is None:
        for i in idx:
            feature_rows.append({"analysis_id": s3_03_input.at[i, "analysis_id"]})
        continue
    times = g["일시"].to_numpy(dtype="datetime64[ns]")
    arrays = {col: g[col].to_numpy(dtype="float64") for col in ["습도_pct", "풍속_m_s", "현지기압_hPa"]}
    for i in idx:
        t = np.datetime64(s3_03_input.at[i, "기준시각"], "ns")
        rec = {"analysis_id": s3_03_input.at[i, "analysis_id"]}
        for d in rapid_defs:
            recent = agg_interval(arrays[d["source"]], times, t, d["recent_start_h"], d["recent_end_h"], d["recent_agg"])
            prev = agg_interval(arrays[d["source"]], times, t, d["prev_start_h"], d["prev_end_h"], d["prev_agg"])
            val = np.nan if pd.isna(recent) or pd.isna(prev) else recent - prev
            rec[d["feature"]] = abs(val) if d["risk_direction"] == "two_sided" and pd.notna(val) else val
            rec[d["feature"] + "_signed"] = val
            rec[d["feature"] + "_recent"] = recent
            rec[d["feature"] + "_previous"] = prev
        feature_rows.append(rec)

rapid_features = pd.DataFrame(feature_rows)
features_s303 = s3_03_input.merge(rapid_features, on="analysis_id", how="left")

fwi_idx = fwi_daily_s303[["기상셀ID", "날짜", "FFMC", "ISI", "FWI"]].copy()
cur = fwi_idx.rename(columns={"날짜": "canadian_ref_date", "FFMC": "FFMC_asof", "ISI": "ISI_asof", "FWI": "FWI_asof"})
prev = fwi_idx.rename(columns={"날짜": "canadian_prev_date", "FFMC": "FFMC_prev1d", "ISI": "ISI_prev1d", "FWI": "FWI_prev1d"})
features_s303 = features_s303.merge(cur, on=["기상셀ID", "canadian_ref_date"], how="left", validate="many_to_one")
features_s303 = features_s303.merge(prev, on=["기상셀ID", "canadian_prev_date"], how="left", validate="many_to_one")
for v in ["FFMC", "ISI", "FWI"]:
    features_s303[f"{v}_change_1d"] = features_s303[f"{v}_asof"] - features_s303[f"{v}_prev1d"]

index_meta = pd.DataFrame([
    {"feature": "FFMC_change_1d", "variable": "canadian_change", "source": "캐나다_FWI_일단위.csv", "recent_start_h": np.nan, "recent_end_h": np.nan, "recent_agg": "asof_minus_prev_day", "prev_start_h": np.nan, "prev_end_h": np.nan, "prev_agg": "asof_minus_prev_day", "risk_direction": "higher", "unit": "index"},
    {"feature": "ISI_change_1d", "variable": "canadian_change", "source": "캐나다_FWI_일단위.csv", "recent_start_h": np.nan, "recent_end_h": np.nan, "recent_agg": "asof_minus_prev_day", "prev_start_h": np.nan, "prev_end_h": np.nan, "prev_agg": "asof_minus_prev_day", "risk_direction": "higher", "unit": "index"},
    {"feature": "FWI_change_1d", "variable": "canadian_change", "source": "캐나다_FWI_일단위.csv", "recent_start_h": np.nan, "recent_end_h": np.nan, "recent_agg": "asof_minus_prev_day", "prev_start_h": np.nan, "prev_end_h": np.nan, "prev_agg": "asof_minus_prev_day", "risk_direction": "higher", "unit": "index"},
])
feature_meta_s303 = pd.concat([rapid_meta, index_meta], ignore_index=True, sort=False)
feature_list_s303 = feature_meta_s303["feature"].tolist()

missing_rows = []
for feature in feature_list_s303:
    for group_name, sub in features_s303.groupby("sample_group", dropna=False):
        missing_rows.append({"feature": feature, "sample_group": group_name, "row_n": len(sub), "missing_n": int(sub[feature].isna().sum()), "missing_pct": float(sub[feature].isna().mean() * 100)})
missingness_s303 = pd.DataFrame(missing_rows).merge(feature_meta_s303, on="feature", how="left")

fire_s303 = features_s303[features_s303["target"].eq(1)].copy()
control_s303 = features_s303[features_s303["target"].eq(0)].copy()
control_means_s303 = control_s303.groupby("fire_exposure_id", as_index=False)[feature_list_s303].mean(numeric_only=True)
control_counts_s303 = control_s303.groupby("fire_exposure_id", as_index=False).agg(control_row_n=("analysis_id", "count"))
paired_s303 = fire_s303.merge(control_means_s303, on="fire_exposure_id", how="left", suffixes=("_fire", "_control_mean"))
paired_s303 = paired_s303.merge(control_counts_s303, on="fire_exposure_id", how="left")

delta_rows = []
for _, meta in feature_meta_s303.iterrows():
    feature = meta["feature"]
    tmp = paired_s303[["fire_exposure_id", "기상셀ID", "기준시각", "기후지형유형", "date", "month", "hour", "matched_control_n", "full_5_matched", "is_ys0071", "is_less_than_5_matched", "is_2021_feb_cluster", "control_row_n", f"{feature}_fire", f"{feature}_control_mean"]].copy()
    tmp = tmp.rename(columns={f"{feature}_fire": "fire_value", f"{feature}_control_mean": "control_mean"})
    tmp["feature"] = feature
    tmp["variable"] = meta["variable"]
    tmp["risk_direction"] = meta["risk_direction"]
    tmp["unit"] = meta["unit"]
    tmp["delta"] = tmp["fire_value"] - tmp["control_mean"]
    tmp["risk_direction_match"] = np.where(tmp["risk_direction"].eq("lower"), tmp["delta"] < 0, tmp["delta"] > 0)
    delta_rows.append(tmp)
delta_raw_s303 = pd.concat(delta_rows, ignore_index=True)

population_audit_s303 = pd.DataFrame([
    {"item": "analysis_unit", "value": "fire_exposure_id paired rapid-change delta"},
    {"item": "fire_exposure_n", "value": int(fire_s303["fire_exposure_id"].nunique())},
    {"item": "fire_row_n", "value": int(len(fire_s303))},
    {"item": "control_row_n", "value": int(len(control_s303))},
    {"item": "paired_fire_exposure_n", "value": int(paired_s303["fire_exposure_id"].nunique())},
    {"item": "delta_raw_row_n", "value": int(len(delta_raw_s303))},
    {"item": "analysis_id_duplicate_n", "value": int(features_s303["analysis_id"].duplicated().sum())},
    {"item": "fire_exposure_duplicate_in_fire_n", "value": int(fire_s303["fire_exposure_id"].duplicated().sum())},
    {"item": "control_without_fire_id_n", "value": int(control_s303["fire_exposure_id"].isna().sum())},
    {"item": "canadian_asof_future_violation_n", "value": int((s3_03_input["canadian_ref_time"] > s3_03_input["기준시각"]).sum())},
])

write_table(S3_03_SCOPE, "S3-03_analysis_scope.csv")
write_table(population_audit_s303, "S3-03_population_audit.csv")
write_table(feature_meta_s303, "S3-03_rapid_change_definitions.csv")
write_table(missingness_s303, "S3-03_feature_missingness_audit.csv")
write_table(delta_raw_s303, "S3-03_rapid_change_delta_raw.csv")
print(population_audit_s303)
print(missingness_s303.groupby("sample_group")["missing_pct"].max())


## S3-03 : 대응 delta 및 threshold 조건 요약

`fire_exposure_id`별 급변 프록시 delta와 threshold 조건 충족률, 비율비, p/q값, bootstrap CI, 효과크기를 계산한다.


In [19]:
BOOT_N_S303 = 500
rng_s303 = np.random.default_rng(303)

def bh_fdr_s303(p_values):
    p = np.asarray(p_values, dtype=float)
    q = np.full_like(p, np.nan, dtype=float)
    valid = np.isfinite(p)
    if valid.sum() == 0:
        return q
    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    m = len(ranked)
    adj = ranked * m / np.arange(1, m + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0, 1)
    valid_idx = np.where(valid)[0]
    q[valid_idx[order]] = adj
    return q

def bootstrap_ci_s303(values, blocks, reps=BOOT_N_S303):
    vals = np.asarray(values, dtype=float)
    block_arr = np.asarray(blocks)
    ok = np.isfinite(vals) & pd.notna(block_arr)
    vals = vals[ok]
    block_arr = block_arr[ok]
    if len(vals) < 2:
        return np.nan, np.nan
    unique_blocks = pd.unique(block_arr)
    if len(unique_blocks) < 2:
        return np.nan, np.nan
    by_block = {b: vals[block_arr == b] for b in unique_blocks}
    means = np.empty(reps)
    for r in range(reps):
        sample_blocks = rng_s303.choice(unique_blocks, size=len(unique_blocks), replace=True)
        sample_vals = np.concatenate([by_block[b] for b in sample_blocks])
        means[r] = np.nanmean(sample_vals)
    return float(np.nanpercentile(means, 2.5)), float(np.nanpercentile(means, 97.5))

def summarize_delta_s303(sub, group_label):
    rows = []
    meta_lookup = feature_meta_s303.set_index("feature")
    for feature, g in sub.groupby("feature", sort=False):
        meta = meta_lookup.loc[feature]
        complete = g[g["delta"].notna()].copy()
        x = complete["delta"].to_numpy(dtype=float)
        n = len(x)
        if n == 0:
            continue
        mean_delta = float(np.mean(x))
        sd_delta = float(np.std(x, ddof=1)) if n > 1 else np.nan
        dz = mean_delta / sd_delta if sd_delta and np.isfinite(sd_delta) and sd_delta != 0 else np.nan
        se = sd_delta / math.sqrt(n) if n > 1 and np.isfinite(sd_delta) else np.nan
        ci_low = mean_delta - stats.t.ppf(0.975, n - 1) * se if n > 1 and np.isfinite(se) else np.nan
        ci_high = mean_delta + stats.t.ppf(0.975, n - 1) * se if n > 1 and np.isfinite(se) else np.nan
        try:
            t_p = float(stats.ttest_1samp(x, 0, nan_policy="omit").pvalue) if n > 1 else np.nan
        except Exception:
            t_p = np.nan
        try:
            nonzero = x[x != 0]
            w_p = float(stats.wilcoxon(nonzero).pvalue) if len(nonzero) > 0 else np.nan
        except Exception:
            w_p = np.nan
        date_ci_low, date_ci_high = bootstrap_ci_s303(x, complete["date"].to_numpy())
        cell_ci_low, cell_ci_high = bootstrap_ci_s303(x, complete["기상셀ID"].to_numpy())
        risk_direction = meta["risk_direction"]
        direction_pct = float(((x < 0) if risk_direction == "lower" else (x > 0)).mean() * 100)
        rows.append({
            "group": group_label, "feature": feature, "variable": meta["variable"], "risk_direction": risk_direction, "unit": meta["unit"],
            "n": int(n), "fire_mean": float(complete["fire_value"].mean()), "control_mean": float(complete["control_mean"].mean()), "mean_delta": mean_delta, "median_delta": float(np.median(x)), "sd_delta": sd_delta,
            "mean_delta_ci_low_t": ci_low, "mean_delta_ci_high_t": ci_high, "date_boot_ci_low": date_ci_low, "date_boot_ci_high": date_ci_high, "cell_boot_ci_low": cell_ci_low, "cell_boot_ci_high": cell_ci_high,
            "cohen_dz": dz, "direction_match_pct": direction_pct, "paired_t_p": t_p, "wilcoxon_p": w_p, "date_n": int(complete["date"].nunique()), "cell_n": int(complete["기상셀ID"].nunique()),
            "support_flag": "ok" if (n >= 30 and complete["date"].nunique() >= 10 and complete["기상셀ID"].nunique() >= 5) else "sparse_do_not_interpret",
        })
    return pd.DataFrame(rows)

summary_parts_s303 = [summarize_delta_s303(delta_raw_s303, "전체")]
for climate_type, sub in delta_raw_s303.groupby("기후지형유형", sort=False):
    summary_parts_s303.append(summarize_delta_s303(sub, climate_type))
delta_summary_s303 = pd.concat(summary_parts_s303, ignore_index=True)
delta_summary_s303["paired_t_q"] = bh_fdr_s303(delta_summary_s303["paired_t_p"])
delta_summary_s303["wilcoxon_q"] = bh_fdr_s303(delta_summary_s303["wilcoxon_p"])
delta_summary_s303["date_ci_excludes_zero"] = (delta_summary_s303["date_boot_ci_low"] > 0) | (delta_summary_s303["date_boot_ci_high"] < 0)
delta_summary_s303["cell_ci_excludes_zero"] = (delta_summary_s303["cell_boot_ci_low"] > 0) | (delta_summary_s303["cell_boot_ci_high"] < 0)
delta_summary_s303["candidate_priority"] = np.select([
    delta_summary_s303["support_flag"].ne("ok"),
    delta_summary_s303["date_ci_excludes_zero"] & delta_summary_s303["cell_ci_excludes_zero"] & delta_summary_s303["direction_match_pct"].ge(60),
    delta_summary_s303["date_ci_excludes_zero"] & delta_summary_s303["direction_match_pct"].ge(55),
], ["drop_sparse", "strong_rapid_candidate", "weak_rapid_candidate"], default="descriptive_only")

condition_defs = [
    {"condition": "rh_mean_3h_drop_le_m5", "feature": "rh_mean_3h_minus_prev21h_mean", "operator": "<=", "threshold": -5.0, "description": "최근3h 평균습도 - 이전21h 평균습도 <= -5%p"},
    {"condition": "rh_mean_3h_drop_le_m10", "feature": "rh_mean_3h_minus_prev21h_mean", "operator": "<=", "threshold": -10.0, "description": "최근3h 평균습도 - 이전21h 평균습도 <= -10%p"},
    {"condition": "rh_mean_6h_drop_le_m5", "feature": "rh_mean_6h_minus_prev18h_mean", "operator": "<=", "threshold": -5.0, "description": "최근6h 평균습도 - 이전18h 평균습도 <= -5%p"},
    {"condition": "rh_mean_6h_drop_le_m10", "feature": "rh_mean_6h_minus_prev18h_mean", "operator": "<=", "threshold": -10.0, "description": "최근6h 평균습도 - 이전18h 평균습도 <= -10%p"},
    {"condition": "rh_min_6h_drop_le_m10", "feature": "rh_min_6h_minus_prev18h_mean", "operator": "<=", "threshold": -10.0, "description": "최근6h 최소습도 - 이전18h 평균습도 <= -10%p"},
    {"condition": "wind_mean_3h_inc_ge_0p5", "feature": "wind_mean_3h_minus_prev21h_mean", "operator": ">=", "threshold": 0.5, "description": "최근3h 평균풍속 - 이전21h 평균풍속 >= 0.5m/s"},
    {"condition": "wind_max_6h_inc_ge_1p0", "feature": "wind_max_6h_minus_prev18h_max", "operator": ">=", "threshold": 1.0, "description": "최근6h 최대풍속 - 이전18h 최대풍속 >= 1.0m/s"},
    {"condition": "pressure_abs_change_3h_ge_1p0", "feature": "pressure_mean_3h_minus_prev3h_mean", "operator": ">=", "threshold": 1.0, "description": "최근3h와 이전3h 평균 현지기압 차이 절댓값 >= 1.0hPa"},
    {"condition": "ffmc_1d_change_ge_3", "feature": "FFMC_change_1d", "operator": ">=", "threshold": 3.0, "description": "as-of FFMC - 전일 FFMC >= 3"},
    {"condition": "isi_1d_change_ge_0p5", "feature": "ISI_change_1d", "operator": ">=", "threshold": 0.5, "description": "as-of ISI - 전일 ISI >= 0.5"},
]
condition_defs_df = pd.DataFrame(condition_defs)
condition_feature_frames = []
for d in condition_defs:
    vals = features_s303[d["feature"]]
    flag = vals <= d["threshold"] if d["operator"] == "<=" else vals >= d["threshold"]
    flag_numeric = np.where(vals.notna(), flag.astype(float), np.nan)
    condition_feature_frames.append(pd.DataFrame({"analysis_id": features_s303["analysis_id"], d["condition"]: flag_numeric}))
condition_flags = condition_feature_frames[0]
for f in condition_feature_frames[1:]:
    condition_flags = condition_flags.merge(f, on="analysis_id", how="left")
features_conditions = features_s303[["analysis_id", "fire_exposure_id", "sample_group", "target", "기상셀ID", "기준시각", "기후지형유형", "date", "month", "hour"]].merge(condition_flags, on="analysis_id", how="left")

def summarize_condition(cond, group_label, fire_ids=None):
    sub = features_conditions if fire_ids is None else features_conditions[features_conditions["fire_exposure_id"].isin(fire_ids)]
    fire = sub[sub["target"].eq(1)][["fire_exposure_id", "기상셀ID", "date", "기후지형유형", cond]].copy()
    ctrl = sub[sub["target"].eq(0)][["fire_exposure_id", cond]].copy()
    fire[cond] = pd.to_numeric(fire[cond], errors="coerce")
    ctrl[cond] = pd.to_numeric(ctrl[cond], errors="coerce")
    ctrl_mean = ctrl.groupby("fire_exposure_id", as_index=False)[cond].mean().rename(columns={cond: "control_rate_within_match"})
    paired_cond = fire.merge(ctrl_mean, on="fire_exposure_id", how="left")
    paired_cond = paired_cond.dropna(subset=[cond, "control_rate_within_match"])
    diff = paired_cond[cond].astype(float) - paired_cond["control_rate_within_match"].astype(float)
    n = len(paired_cond)
    fire_rate = float(paired_cond[cond].mean()) if n else np.nan
    control_rate = float(paired_cond["control_rate_within_match"].mean()) if n else np.nan
    rd = float(diff.mean()) if n else np.nan
    lo, hi = bootstrap_ci_s303(diff.to_numpy(), paired_cond["date"].to_numpy()) if n else (np.nan, np.nan)
    try:
        p = float(stats.ttest_1samp(diff, 0, nan_policy="omit").pvalue) if n > 1 else np.nan
    except Exception:
        p = np.nan
    pr = (fire_rate + 1e-9) / (control_rate + 1e-9) if pd.notna(fire_rate) and pd.notna(control_rate) else np.nan
    odds_fire = (fire_rate + 0.5 / max(n, 1)) / (1 - fire_rate + 0.5 / max(n, 1)) if pd.notna(fire_rate) else np.nan
    odds_control = (control_rate + 0.5 / max(n, 1)) / (1 - control_rate + 0.5 / max(n, 1)) if pd.notna(control_rate) else np.nan
    return {"group": group_label, "condition": cond, "n": int(n), "fire_condition_rate": fire_rate, "control_condition_rate": control_rate, "matched_risk_diff": rd, "date_boot_ci_low": lo, "date_boot_ci_high": hi, "matched_prevalence_ratio": pr, "matched_or_approx": odds_fire / odds_control if odds_control else np.nan, "paired_t_p": p, "date_n": int(paired_cond["date"].nunique()) if n else 0, "cell_n": int(paired_cond["기상셀ID"].nunique()) if n else 0}

condition_rows = []
for cond in condition_defs_df["condition"]:
    condition_rows.append(summarize_condition(cond, "전체"))
    for climate_type, ids in fire_s303.groupby("기후지형유형")["fire_exposure_id"]:
        condition_rows.append(summarize_condition(cond, climate_type, set(ids)))
threshold_summary_s303 = pd.DataFrame(condition_rows).merge(condition_defs_df, on="condition", how="left")
threshold_summary_s303["paired_t_q"] = bh_fdr_s303(threshold_summary_s303["paired_t_p"])
threshold_summary_s303["support_flag"] = np.where((threshold_summary_s303["n"] >= 30) & (threshold_summary_s303["date_n"] >= 10) & (threshold_summary_s303["cell_n"] >= 5), "ok", "sparse_do_not_interpret")

write_table(delta_summary_s303, "S3-03_rapid_change_delta_summary.csv")
write_table(delta_summary_s303[delta_summary_s303["group"].ne("전체")], "S3-03_rapid_change_by_climate_type.csv")
write_table(condition_defs_df, "S3-03_threshold_definitions.csv")
write_table(threshold_summary_s303, "S3-03_threshold_condition_summary.csv")
write_table(features_conditions, "S3-03_threshold_condition_flags.csv")
print(delta_summary_s303[delta_summary_s303["group"].eq("전체")].sort_values("cohen_dz", key=lambda s: s.abs(), ascending=False)[["feature", "n", "mean_delta", "date_boot_ci_low", "date_boot_ci_high", "cell_boot_ci_low", "cell_boot_ci_high", "cohen_dz", "direction_match_pct", "paired_t_p", "paired_t_q", "candidate_priority"]].head(12))
print(threshold_summary_s303[threshold_summary_s303["group"].eq("전체")].sort_values("matched_risk_diff", key=lambda s: s.abs(), ascending=False)[["condition", "n", "fire_condition_rate", "control_condition_rate", "matched_risk_diff", "date_boot_ci_low", "date_boot_ci_high", "matched_prevalence_ratio", "matched_or_approx", "paired_t_p", "paired_t_q"]].head(10))


## S3-03 : 급변 프록시 핵심 플롯 저장

급변량 분포, paired slope, 층화 forest plot, threshold heatmap을 각각 단일 플롯 파일로 저장한다.


In [21]:
plot_paths_s303 = []
plot_groups = ["전체", "영동 해안형", "영서 내륙형", "고지·산간형"]
plot_palette = {"전체": "#222222", "영동 해안형": "#1b9e77", "영서 내륙형": "#377eb8", "고지·산간형": "#d95f02"}

dist_feature = "rh_mean_3h_minus_prev21h_mean"
dist_df = features_s303[["sample_group", "기후지형유형", dist_feature]].dropna().copy()
plt.figure(figsize=(8.2, 5.0))
sns.ecdfplot(data=dist_df, x=dist_feature, hue="sample_group", hue_order=["fire", "matched_control"], palette={"fire": "#d95f02", "matched_control": "#4c78a8"}, linewidth=2)
plt.axvline(-5, color="#666666", linestyle="--", linewidth=1)
plt.axvline(-10, color="#999999", linestyle=":", linewidth=1)
plt.xlabel("Recent 3h mean RH - previous 21h mean RH (%p)")
plt.ylabel("ECDF")
plt.title("S3-03 rapid humidity change distribution")
plot_paths_s303.append(save_current_plot("S3-03_rapid_change_distribution.png"))

slope_feature = "rh_mean_3h_minus_prev21h_mean"
slope = paired_s303[["fire_exposure_id", "기후지형유형", f"{slope_feature}_fire", f"{slope_feature}_control_mean"]].dropna().copy()
slope = slope.rename(columns={f"{slope_feature}_fire": "fire_value", f"{slope_feature}_control_mean": "control_mean"})
plt.figure(figsize=(7.2, 5.2))
for _, r in slope.iterrows():
    plt.plot([0, 1], [r["control_mean"], r["fire_value"]], color="#777777", alpha=0.05, linewidth=0.8)
med = slope[["control_mean", "fire_value"]].median()
plt.plot([0, 1], [med["control_mean"], med["fire_value"]], color="#d95f02", linewidth=3, marker="o", label="median")
plt.xticks([0, 1], ["matched control mean", "fire"])
plt.ylabel("Recent 3h mean RH - previous 21h mean RH (%p)")
plt.title("S3-03 paired slope: rapid humidity change")
plt.legend(frameon=True)
plot_paths_s303.append(save_current_plot("S3-03_paired_slope_rh_mean_3h_drop.png"))

forest_features = ["rh_mean_3h_minus_prev21h_mean", "rh_mean_6h_minus_prev18h_mean", "rh_min_6h_minus_prev18h_mean", "wind_mean_3h_minus_prev21h_mean", "wind_max_6h_minus_prev18h_max", "pressure_mean_3h_minus_prev3h_mean", "FFMC_change_1d", "ISI_change_1d"]
forest = delta_summary_s303[(delta_summary_s303["feature"].isin(forest_features)) & (delta_summary_s303["group"].isin(plot_groups))].copy()
forest["label"] = forest["group"] + " | " + forest["feature"]
forest = forest.sort_values("cohen_dz")
plt.figure(figsize=(9.5, max(6, 0.24 * len(forest) + 1.6)))
y = np.arange(len(forest))
scaled_low = (forest["mean_delta"] - forest["date_boot_ci_low"]).abs() / forest["sd_delta"].replace(0, np.nan)
scaled_high = (forest["date_boot_ci_high"] - forest["mean_delta"]).abs() / forest["sd_delta"].replace(0, np.nan)
plt.errorbar(forest["cohen_dz"], y, xerr=[scaled_low, scaled_high], fmt="o", color="#333333", ecolor="#888888", capsize=3)
plt.axvline(0, color="#666666", linewidth=1)
plt.yticks(y, forest["label"])
plt.xlabel("Cohen's dz (date-bootstrap CI scaled by delta SD)")
plt.ylabel("")
plt.title("S3-03 rapid-change paired delta by climate type")
plot_paths_s303.append(save_current_plot("S3-03_rapid_change_by_type_forest.png"))

heat = threshold_summary_s303[threshold_summary_s303["support_flag"].eq("ok")].copy()
heat["group"] = pd.Categorical(heat["group"], categories=plot_groups, ordered=True)
piv = heat.pivot_table(index="condition", columns="group", values="matched_risk_diff", aggfunc="mean")
plt.figure(figsize=(8.2, max(5.0, 0.35 * len(piv) + 1.5)))
sns.heatmap(piv, center=0, cmap="vlag", annot=True, fmt=".3f", linewidths=0.4, cbar_kws={"label": "fire rate - matched control rate"})
plt.xlabel("")
plt.ylabel("")
plt.title("S3-03 threshold condition matched risk difference")
plot_paths_s303.append(save_current_plot("S3-03_threshold_condition_heatmap.png"))

plot_manifest_s303 = pd.DataFrame([{"plot_path": str(p.relative_to(REPO_ROOT)), "file_name": p.name} for p in plot_paths_s303])
write_table(plot_manifest_s303, "S3-03_plot_manifest.csv")
artifact_manifest_s303 = pd.DataFrame([{ "artifact_type": "table", "path": str((TABLE_DIR / name).relative_to(REPO_ROOT)) } for name in ["S3-03_analysis_scope.csv", "S3-03_population_audit.csv", "S3-03_rapid_change_definitions.csv", "S3-03_feature_missingness_audit.csv", "S3-03_rapid_change_delta_raw.csv", "S3-03_rapid_change_delta_summary.csv", "S3-03_rapid_change_by_climate_type.csv", "S3-03_threshold_definitions.csv", "S3-03_threshold_condition_summary.csv", "S3-03_threshold_condition_flags.csv", "S3-03_plot_manifest.csv"]] + [{"artifact_type": "plot", "path": str(p.relative_to(REPO_ROOT))} for p in plot_paths_s303])
write_table(artifact_manifest_s303, "S3-03_artifact_manifest.csv")
print(plot_manifest_s303)


## S3-04 : 지속 건조·무강수·마지막 강수 후 경과

단기 건조는 순간값인가, 무강수 지속·강수 후 경과와 함께 나타나는가? 마지막 유의 강수 후 경과시간, 무강수 지속시간, 강수 후 습도회복/재건조 속도 및 FFMC 회복을 계산하고 ECDF와 층화 플롯을 저장한다. 유의 강수 기준은 0.1mm, 1.0mm, 5.0mm로 나누어 분석한다.

In [23]:
# S3-04 : 지속 건조·무강수·마지막 강수 후 경과
from scipy import stats

raw_weather_path = REPO_ROOT / "data" / "강원도_날씨데이터" / "강원도날씨_격자_시간단위.csv"
fwi_path = DATA_DIR / "캐나다_FWI_일단위.csv"
if not raw_weather_path.exists():
    raise FileNotFoundError(raw_weather_path)
if not fwi_path.exists():
    raise FileNotFoundError(fwi_path)

if "analysis_keys" not in globals():
    raise RuntimeError("S3-01 analysis_keys가 필요합니다. 노트북을 처음부터 실행하세요.")
if analysis_keys["analysis_id"].duplicated().any():
    raise ValueError("analysis_id duplicated before S3-04")

s3_04_input = analysis_keys.copy()
s3_04_input["기준시각"] = pd.to_datetime(s3_04_input["기준시각"])
s3_04_input["date"] = s3_04_input["기준시각"].dt.date.astype(str)

print("Loading weather data for S3-04...")
raw_weather_s304 = read_csv_kr(
    raw_weather_path,
    usecols=["기상셀ID", "일시", "습도_pct", "풍속_m_s", "강수량_mm"],
    parse_dates=["일시"],
).sort_values(["기상셀ID", "일시"]).reset_index(drop=True)

print("Loading daily FWI data for S3-04...")
fwi_daily_s304 = read_csv_kr(fwi_path, parse_dates=["날짜"])
fwi_daily_s304["날짜"] = fwi_daily_s304["날짜"].dt.date
fwi_lookup = {}
for cell, date, ffmc in zip(fwi_daily_s304["기상셀ID"], fwi_daily_s304["날짜"], fwi_daily_s304["FFMC"]):
    fwi_lookup[(cell, date)] = ffmc

print("Grouping weather data for S3-04...")
weather_groups_s304 = {cell: g for cell, g in raw_weather_s304.groupby("기상셀ID", sort=False)}

print("Computing variables for S3-04...")
thresholds = [0.1, 1.0, 5.0]
results_s304 = []

for idx, row in s3_04_input.iterrows():
    cell = row["기상셀ID"]
    t = row["기준시각"]
    t_np = np.datetime64(t, "ns")
    
    g = weather_groups_s304.get(cell)
    if g is None:
        rec = {"analysis_id": row["analysis_id"]}
        for thr in thresholds:
            thr_str = str(thr).replace(".", "p")
            rec[f"last_rain_elapsed_h_{thr_str}"] = np.nan
            rec[f"dry_spell_h_{thr_str}"] = np.nan
            rec[f"min_rh_post_{thr_str}"] = np.nan
            rec[f"drying_rate_{thr_str}"] = np.nan
            rec[f"drying_amount_{thr_str}"] = np.nan
            rec[f"drying_rate_to_min_{thr_str}"] = np.nan
            rec[f"rain_to_ffmc_recovery_{thr_str}"] = np.nan
        rec["rain_sum_72h"] = np.nan
        rec["rh_drop_12h"] = np.nan
        rec["wind_inc_12h"] = np.nan
        rec["wind_max_12h"] = np.nan
        rec["post_rain_drying_proxy"] = np.nan
        results_s304.append(rec)
        continue
        
    times = g["일시"].to_numpy(dtype="datetime64[ns]")
    rain = g["강수량_mm"].to_numpy(dtype="float64")
    rh = g["습도_pct"].to_numpy(dtype="float64")
    wind = g["풍속_m_s"].to_numpy(dtype="float64")
    
    end_idx = np.searchsorted(times, t_np, side="left")
    
    rec = {"analysis_id": row["analysis_id"]}
    
    # Current as-of FFMC
    canadian_ref_date = (t - pd.to_timedelta(int(t.hour < 12), unit="D")).date()
    ffmc_current = fwi_lookup.get((cell, canadian_ref_date), np.nan)
    
    # Recent 72h rain
    start_72h = np.searchsorted(times, t_np - np.timedelta64(72, "h"), side="left")
    rain_sum_72h = np.sum(rain[start_72h:end_idx]) if end_idx > start_72h else 0
    
    # Recent 12h window
    start_12h = np.searchsorted(times, t_np - np.timedelta64(12, "h"), side="left")
    rh_window = rh[start_12h:end_idx]
    wind_window = wind[start_12h:end_idx]
    
    if len(rh_window) >= 12:
        rh_recent_3h = np.mean(rh_window[-3:])
        rh_prev_9h = np.mean(rh_window[:-3])
        rh_drop = rh_recent_3h - rh_prev_9h
        
        wind_recent_3h = np.mean(wind_window[-3:])
        wind_prev_9h = np.mean(wind_window[:-3])
        wind_inc = wind_recent_3h - wind_prev_9h
        
        wind_max = np.max(wind_window)
    else:
        rh_drop = np.nan
        wind_inc = np.nan
        wind_max = np.nan
        
    rec["rain_sum_72h"] = rain_sum_72h
    rec["rh_drop_12h"] = rh_drop
    rec["wind_inc_12h"] = wind_inc
    rec["wind_max_12h"] = wind_max
    rec["post_rain_drying_proxy"] = (rain_sum_72h > 0) & (rh_drop <= -5.0) & (wind_max >= 3.0)
    
    for thr in thresholds:
        thr_str = str(thr).replace(".", "p")
        
        j = -1
        for k in range(end_idx - 1, -1, -1):
            if rain[k] >= thr:
                j = k
                break
        
        if j != -1:
            t_rain = times[j]
            t_rain_pd = pd.to_datetime(t_rain)
            elapsed = (t_np - t_rain) / np.timedelta64(1, 'h')
            elapsed_capped = min(elapsed, 720.0)
            
            RH_at_rain = rh[j]
            RH_current = rh[end_idx - 1] if end_idx > 0 else np.nan
            min_rh_post = np.nanmin(rh[j:end_idx]) if j < end_idx else np.nan
            
            drying_amount = RH_at_rain - RH_current
            drying_rate = (RH_at_rain - RH_current) / (elapsed + 1e-5)
            drying_rate_to_min = (RH_at_rain - min_rh_post) / (elapsed + 1e-5)
            
            rain_ref_date = (t_rain_pd - pd.to_timedelta(int(t_rain_pd.hour < 12), unit="D")).date()
            ffmc_at_rain = fwi_lookup.get((cell, rain_ref_date), np.nan)
            ffmc_recovery = ffmc_current - ffmc_at_rain if pd.notna(ffmc_current) and pd.notna(ffmc_at_rain) else np.nan
            
            rec[f"last_rain_elapsed_h_{thr_str}"] = elapsed_capped
            rec[f"dry_spell_h_{thr_str}"] = elapsed_capped
            rec[f"min_rh_post_{thr_str}"] = min_rh_post
            rec[f"drying_rate_{thr_str}"] = drying_rate
            rec[f"drying_amount_{thr_str}"] = drying_amount
            rec[f"drying_rate_to_min_{thr_str}"] = drying_rate_to_min
            rec[f"rain_to_ffmc_recovery_{thr_str}"] = ffmc_recovery
        else:
            rec[f"last_rain_elapsed_h_{thr_str}"] = 720.0
            rec[f"dry_spell_h_{thr_str}"] = 720.0
            rec[f"min_rh_post_{thr_str}"] = np.nan
            rec[f"drying_rate_{thr_str}"] = np.nan
            rec[f"drying_amount_{thr_str}"] = np.nan
            rec[f"drying_rate_to_min_{thr_str}"] = np.nan
            rec[f"rain_to_ffmc_recovery_{thr_str}"] = np.nan

    results_s304.append(rec)

features_s304 = pd.DataFrame(results_s304)
merged_df_s304 = s3_04_input.merge(features_s304, on="analysis_id", how="left")

# Summarize continuous variables
def summarize_continuous_deltas_s304(df_wide, features_to_sum):
    fire_part = df_wide[df_wide["target"] == 1].copy()
    control_part = df_wide[df_wide["target"] == 0].copy()
    
    control_means = control_part.groupby("fire_exposure_id", as_index=False)[features_to_sum].mean()
    control_counts = control_part.groupby("fire_exposure_id", as_index=False).agg(control_row_n=("analysis_id", "count"))
    paired = fire_part.merge(control_means, on="fire_exposure_id", how="left", suffixes=("_fire", "_control_mean"))
    paired = paired.merge(control_counts, on="fire_exposure_id", how="left")
    
    delta_rows = []
    for f in features_to_sum:
        fcol = f"{f}_fire"
        ccol = f"{f}_control_mean"
        tmp = paired[["fire_exposure_id", "기상셀ID", "기준시각", "기후지형유형", "date", "month", "hour", "matched_control_n", "full_5_matched", "is_ys0071", "is_less_than_5_matched", "is_2021_feb_cluster", "control_row_n", fcol, ccol]].copy()
        tmp = tmp.rename(columns={fcol: "fire_value", ccol: "control_mean"})
        tmp["feature"] = f
        tmp["delta"] = tmp["fire_value"] - tmp["control_mean"]
        tmp["risk_direction"] = "higher"
        tmp["risk_direction_match"] = tmp["delta"] > 0
        delta_rows.append(tmp)
    
    delta_long = pd.concat(delta_rows, ignore_index=True)
    
    def get_summary_stats(sub_df, group_label):
        stats_rows = []
        for feat, g in sub_df.groupby("feature", sort=False):
            complete = g[g["delta"].notna()].copy()
            x = complete["delta"].to_numpy(dtype=float)
            n = len(x)
            if n == 0: continue
            
            mean_delta = float(np.mean(x))
            sd_delta = float(np.std(x, ddof=1)) if n > 1 else np.nan
            dz = mean_delta / sd_delta if sd_delta and np.isfinite(sd_delta) and sd_delta != 0 else np.nan
            se = sd_delta / math.sqrt(n) if n > 1 and np.isfinite(sd_delta) else np.nan
            ci_low = mean_delta - stats.t.ppf(0.975, n - 1) * se if n > 1 and np.isfinite(se) else np.nan
            ci_high = mean_delta + stats.t.ppf(0.975, n - 1) * se if n > 1 and np.isfinite(se) else np.nan
            
            try: t_p = float(stats.ttest_1samp(x, 0, nan_policy="omit").pvalue) if n > 1 else np.nan
            except Exception: t_p = np.nan
            try:
                nonzero = x[x != 0]
                w_p = float(stats.wilcoxon(nonzero).pvalue) if len(nonzero) > 0 else np.nan
            except Exception: w_p = np.nan
            
            date_ci_low, date_ci_high = bootstrap_ci(x, complete["date"].to_numpy())
            cell_ci_low, cell_ci_high = bootstrap_ci(x, complete["기상셀ID"].to_numpy())
            
            direction_pct = float((x > 0).mean() * 100)
            
            stats_rows.append({
                "group": group_label,
                "feature": feat,
                "n": int(n),
                "fire_mean": float(complete["fire_value"].mean()),
                "control_mean": float(complete["control_mean"].mean()),
                "mean_delta": mean_delta,
                "median_delta": float(np.median(x)),
                "sd_delta": sd_delta,
                "mean_delta_ci_low_t": ci_low,
                "mean_delta_ci_high_t": ci_high,
                "date_boot_ci_low": date_ci_low,
                "date_boot_ci_high": date_ci_high,
                "cell_boot_ci_low": cell_ci_low,
                "cell_boot_ci_high": cell_ci_high,
                "cohen_dz": dz,
                "direction_match_pct": direction_pct,
                "paired_t_p": t_p,
                "wilcoxon_p": w_p,
                "date_n": int(complete["date"].nunique()),
                "cell_n": int(complete["기상셀ID"].nunique()),
                "support_flag": "ok" if (n >= 30 and complete["date"].nunique() >= 10 and complete["기상셀ID"].nunique() >= 5) else "sparse_do_not_interpret"
            })
        return pd.DataFrame(stats_rows)

    summary_parts = [get_summary_stats(delta_long, "전체")]
    for ct, sub in delta_long.groupby("기후지형유형", sort=False):
        summary_parts.append(get_summary_stats(sub, ct))
    summary_df = pd.concat(summary_parts, ignore_index=True)
    summary_df["paired_t_q"] = bh_fdr(summary_df["paired_t_p"])
    summary_df["wilcoxon_q"] = bh_fdr(summary_df["wilcoxon_p"])
    return summary_df

features_to_sum_s304 = [
    "last_rain_elapsed_h_0p1", "last_rain_elapsed_h_1p0", "last_rain_elapsed_h_5p0",
    "drying_amount_0p1", "drying_amount_1p0", "drying_amount_5p0",
    "drying_rate_0p1", "drying_rate_1p0", "drying_rate_5p0",
    "rain_to_ffmc_recovery_0p1", "rain_to_ffmc_recovery_1p0", "rain_to_ffmc_recovery_5p0"
]

print("Summarizing continuous variables...")
rain_elapsed_summary = summarize_continuous_deltas_s304(merged_df_s304, features_to_sum_s304)
write_table(rain_elapsed_summary, "S3-04_rain_elapsed_summary.csv")

# Summarize binary conditions
def summarize_binary_conditions_s304(df_wide, condition_specs):
    df_conds = df_wide[["analysis_id", "fire_exposure_id", "sample_group", "target", "기상셀ID", "기준시각", "기후지형유형", "date"]].copy()
    for name, feat, op, val in condition_specs:
        if op == ">":
            df_conds[name] = np.where(df_wide[feat].notna(), (df_wide[feat] > val).astype(float), np.nan)
        elif op == "<=":
            df_conds[name] = np.where(df_wide[feat].notna(), (df_wide[feat] <= val).astype(float), np.nan)
            
    def summarize_single_condition(cond, group_label, subset_exposure_ids=None):
        sub = df_conds if subset_exposure_ids is None else df_conds[df_conds["fire_exposure_id"].isin(subset_exposure_ids)]
        
        fire = sub[sub["target"] == 1][["fire_exposure_id", "기상셀ID", "date", cond]].copy()
        ctrl = sub[sub["target"] == 0][["fire_exposure_id", cond]].copy()
        
        ctrl_mean = ctrl.groupby("fire_exposure_id", as_index=False)[cond].mean().rename(columns={cond: "control_rate"})
        paired = fire.merge(ctrl_mean, on="fire_exposure_id", how="left")
        paired = paired.dropna(subset=[cond, "control_rate"])
        
        diff = paired[cond] - paired["control_rate"]
        n = len(paired)
        if n == 0: return {}
        
        fire_rate = float(paired[cond].mean())
        control_rate = float(paired["control_rate"].mean())
        rd = float(diff.mean())
        lo, hi = bootstrap_ci(diff.to_numpy(), paired["date"].to_numpy())
        
        try: p = float(stats.ttest_1samp(diff, 0, nan_policy="omit").pvalue) if n > 1 else np.nan
        except Exception: p = np.nan
        
        pr = (fire_rate + 1e-9) / (control_rate + 1e-9)
        odds_fire = (fire_rate + 0.5 / n) / (1.0 - fire_rate + 0.5 / n)
        odds_control = (control_rate + 0.5 / n) / (1.0 - control_rate + 0.5 / n)
        or_val = odds_fire / odds_control if odds_control else np.nan
        
        return {
            "group": group_label,
            "condition": cond,
            "n": int(n),
            "fire_condition_rate": fire_rate,
            "control_condition_rate": control_rate,
            "matched_risk_diff": rd,
            "date_boot_ci_low": lo,
            "date_boot_ci_high": hi,
            "matched_prevalence_ratio": pr,
            "matched_or_approx": or_val,
            "paired_t_p": p,
            "date_n": int(paired["date"].nunique()),
            "cell_n": int(paired["기상셀ID"].nunique()),
            "support_flag": "ok" if (n >= 30 and paired["date"].nunique() >= 10 and paired["기상셀ID"].nunique() >= 5) else "sparse_do_not_interpret"
        }

    cond_rows = []
    for spec in condition_specs:
        cond_name = spec[0]
        cond_rows.append(summarize_single_condition(cond_name, "전체"))
        for ct, ids in df_wide[df_wide["target"] == 1].groupby("기후지형유형")["fire_exposure_id"]:
            res = summarize_single_condition(cond_name, ct, set(ids))
            if res: cond_rows.append(res)
            
    summary_df = pd.DataFrame([r for r in cond_rows if r])
    summary_df["paired_t_q"] = bh_fdr(summary_df["paired_t_p"])
    return summary_df

dry_spell_specs = [
    ("dry_spell_0p1_gt_24h", "dry_spell_h_0p1", ">", 24.0),
    ("dry_spell_0p1_gt_48h", "dry_spell_h_0p1", ">", 48.0),
    ("dry_spell_0p1_gt_72h", "dry_spell_h_0p1", ">", 72.0),
    ("dry_spell_0p1_gt_120h", "dry_spell_h_0p1", ">", 120.0),
    ("dry_spell_0p1_gt_240h", "dry_spell_h_0p1", ">", 240.0),
    ("dry_spell_1p0_gt_24h", "dry_spell_h_1p0", ">", 24.0),
    ("dry_spell_1p0_gt_48h", "dry_spell_h_1p0", ">", 48.0),
    ("dry_spell_1p0_gt_72h", "dry_spell_h_1p0", ">", 72.0),
    ("dry_spell_1p0_gt_120h", "dry_spell_h_1p0", ">", 120.0),
    ("dry_spell_1p0_gt_240h", "dry_spell_h_1p0", ">", 240.0),
    ("dry_spell_5p0_gt_24h", "dry_spell_h_5p0", ">", 24.0),
    ("dry_spell_5p0_gt_48h", "dry_spell_h_5p0", ">", 48.0),
    ("dry_spell_5p0_gt_72h", "dry_spell_h_5p0", ">", 72.0),
    ("dry_spell_5p0_gt_120h", "dry_spell_h_5p0", ">", 120.0),
    ("dry_spell_5p0_gt_240h", "dry_spell_h_5p0", ">", 240.0),
]

print("Summarizing dry spell thresholds...")
dry_spell_threshold_summary = summarize_binary_conditions_s304(merged_df_s304, dry_spell_specs)
write_table(dry_spell_threshold_summary, "S3-04_dry_spell_threshold_summary.csv")

proxy_specs = [
    ("post_rain_drying_proxy", "post_rain_drying_proxy", ">", 0.0),
]
print("Summarizing post-rain drying proxy...")
post_rain_drying_summary = summarize_binary_conditions_s304(merged_df_s304, proxy_specs)
write_table(post_rain_drying_summary, "S3-04_post_rain_drying_proxy.csv")

print("S3-04 calculations and tables completed!")


In [24]:
# S3-04 플롯: ECDF 플롯과 기후유형별 프록시 비율 바차트를 저장한다.
print("Creating S3-04 ECDF plot...")
plt.figure(figsize=(9, 6))
plot_df = merged_df_s304[["sample_group", "last_rain_elapsed_h_0p1", "last_rain_elapsed_h_5p0"]].dropna()

sns.ecdfplot(
    data=plot_df, x="last_rain_elapsed_h_0p1", hue="sample_group", 
    palette={"fire": "#d95f02", "matched_control": "#4c78a8"},
    linewidth=2.5
)
sns.ecdfplot(
    data=plot_df, x="last_rain_elapsed_h_5p0", hue="sample_group", 
    palette={"fire": "#d95f02", "matched_control": "#4c78a8"},
    linewidth=1.8, linestyle="--", alpha=0.7
)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='#d95f02', lw=2.5, label='Fire (0.1mm rain thr)'),
    Line2D([0], [0], color='#4c78a8', lw=2.5, label='Matched Control (0.1mm)'),
    Line2D([0], [0], color='#d95f02', lw=1.8, ls='--', label='Fire (5.0mm rain thr)'),
    Line2D([0], [0], color='#4c78a8', lw=1.8, ls='--', label='Matched Control (5.0mm)'),
]
plt.legend(handles=legend_elements, loc="lower right", frameon=True)
plt.axvline(24, color="#999999", linestyle=":", alpha=0.8)
plt.axvline(72, color="#999999", linestyle=":", alpha=0.8)
plt.axvline(240, color="#999999", linestyle=":", alpha=0.8)
plt.xlabel("Elapsed hours since last rain event (capped at 720h)")
plt.ylabel("ECDF")
plt.title("S3-04 rain elapsed hours ECDF (Fire vs Matched Control)")
ecdf_plot_path = save_current_plot("S3-04_rain_elapsed_ecdf.png")

print("Creating S3-04 post-rain drying proxy plot...")
plt.figure(figsize=(8.5, 5))
proxy_rates = merged_df_s304.groupby(["기후지형유형", "sample_group"])[["post_rain_drying_proxy"]].mean().reset_index()
proxy_rates = proxy_rates.rename(columns={"post_rain_drying_proxy": "proxy_rate"})
sns.barplot(
    data=proxy_rates, x="기후지형유형", y="proxy_rate", hue="sample_group",
    palette={"fire": "#d95f02", "matched_control": "#4c78a8"}
)
plt.ylabel("Prevalence rate of proxy")
plt.xlabel("Climate Type")
plt.title("S3-04 post-rain rapid drying proxy rate by Climate Type")
proxy_plot_path = save_current_plot("S3-04_post_rain_drying_by_type.png")

plot_paths_s304 = [ecdf_plot_path, proxy_plot_path]
plot_manifest_s304 = pd.DataFrame([{"plot_path": str(p.relative_to(REPO_ROOT)), "file_name": p.name} for p in plot_paths_s304])
write_table(plot_manifest_s304, "S3-04_plot_manifest.csv")

artifact_manifest_s304 = pd.DataFrame([
    { "artifact_type": "table", "path": str((TABLE_DIR / name).relative_to(REPO_ROOT)).replace("\\", "/") }
    for name in ["S3-04_rain_elapsed_summary.csv", "S3-04_dry_spell_threshold_summary.csv", "S3-04_post_rain_drying_proxy.csv", "S3-04_plot_manifest.csv"]
] + [
    {"artifact_type": "plot", "path": str(p.relative_to(REPO_ROOT)).replace("\\", "/")}
    for p in plot_paths_s304
])
write_table(artifact_manifest_s304, "S3-04_artifact_manifest.csv")
print("S3-04 plots and artifact manifest completed!")


## S3-05 : 지속·순간 강풍과 풍향

전체에서는 약했던 풍속 신호가 영동·서풍계열·강풍 지속 조건에서 강화되는가? 풍속 지속시간, 최대풍속, 서풍계열×강풍 교차표, 원형 통계를 산출하고 wind rose와 효과크기 forest plot을 저장한다.

In [26]:
# S3-05 : 지속·순간 강풍과 풍향
import math
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

raw_weather_path = REPO_ROOT / "data" / "강원도_날씨데이터" / "강원도날씨_격자_시간단위.csv"
if not raw_weather_path.exists():
    raise FileNotFoundError(raw_weather_path)

if "analysis_keys" not in globals():
    raise RuntimeError("S3-01 analysis_keys가 필요합니다. 노트북을 처음부터 실행하세요.")
if analysis_keys["analysis_id"].duplicated().any():
    raise ValueError("analysis_id duplicated before S3-05")

s3_05_input = analysis_keys.copy()
s3_05_input["기준시각"] = pd.to_datetime(s3_05_input["기준시각"])
s3_05_input["date"] = s3_05_input["기준시각"].dt.date.astype(str)

print("Loading weather data for S3-05...")
raw_weather_s305 = pd.read_csv(
    raw_weather_path,
    usecols=["기상셀ID", "일시", "풍속_m_s", "풍향_deg", "습도_pct"],
    parse_dates=["일시"]
).sort_values(["기상셀ID", "일시"]).reset_index(drop=True)

print("Calculating raw weather wind features...")
raw_weather_s305["풍향_sin"] = np.sin(np.radians(raw_weather_s305["풍향_deg"]))
raw_weather_s305["풍향_cos"] = np.cos(np.radians(raw_weather_s305["풍향_deg"]))
raw_weather_s305["서풍계열_여부"] = ((raw_weather_s305["풍향_deg"] >= 202.5) & (raw_weather_s305["풍향_deg"] <= 292.5)).astype(int)

print("Computing local wind p90...")
raw_weather_s305["month"] = raw_weather_s305["일시"].dt.month
raw_weather_s305["hour"] = raw_weather_s305["일시"].dt.hour
p90_df_s305 = raw_weather_s305.groupby(["기상셀ID", "month", "hour"])["풍속_m_s"].quantile(0.90).reset_index().rename(columns={"풍속_m_s": "local_p90_wind"})
raw_weather_s305 = raw_weather_s305.merge(p90_df_s305, on=["기상셀ID", "month", "hour"], how="left")

weather_groups_s305 = {cell: g for cell, g in raw_weather_s305.groupby("기상셀ID", sort=False)}

print("Computing wind variables for each exposure row...")
results_s305 = []
for idx, row in s3_05_input.iterrows():
    cell = row["기상셀ID"]
    t = row["기준시각"]
    t_np = np.datetime64(t, "ns")
    
    g = weather_groups_s305.get(cell)
    if g is None:
        results_s305.append({
            "analysis_id": row["analysis_id"],
            "wind_duration_3m_s": np.nan,
            "wind_duration_5m_s": np.nan,
            "wind_duration_local_p90": np.nan,
            "westerly_current": np.nan,
            "풍향_sin_current": np.nan,
            "풍향_cos_current": np.nan,
            "wind_mean_6h": np.nan,
            "wind_max_6h": np.nan,
            "wind_mean_24h": np.nan,
            "wind_max_24h": np.nan,
            "westerly_strong_current_3m_s": np.nan,
            "westerly_strong_current_5m_s": np.nan,
            "westerly_strong_max_6h": np.nan,
            "westerly_strong_max_24h": np.nan,
            "시점_풍향_deg": np.nan
        })
        continue
        
    times = g["일시"].to_numpy(dtype="datetime64[ns]")
    wind = g["풍속_m_s"].to_numpy(dtype="float64")
    deg = g["풍향_deg"].to_numpy(dtype="float64")
    sin_arr = g["풍향_sin"].to_numpy(dtype="float64")
    cos_arr = g["풍향_cos"].to_numpy(dtype="float64")
    westerly = g["서풍계열_여부"].to_numpy(dtype="int")
    p90 = g["local_p90_wind"].to_numpy(dtype="float64")
    
    end_idx = np.searchsorted(times, t_np, side="left")
    
    dur_3 = 0
    for k in range(end_idx - 1, -1, -1):
        if wind[k] >= 3.0:
            dur_3 += 1
        else:
            break
            
    dur_5 = 0
    for k in range(end_idx - 1, -1, -1):
        if wind[k] >= 5.0:
            dur_5 += 1
        else:
            break
            
    dur_p90 = 0
    for k in range(end_idx - 1, -1, -1):
        if wind[k] >= p90[k]:
            dur_p90 += 1
        else:
            break
            
    curr_westerly = westerly[end_idx - 1] if end_idx > 0 else np.nan
    curr_sin = sin_arr[end_idx - 1] if end_idx > 0 else np.nan
    curr_cos = cos_arr[end_idx - 1] if end_idx > 0 else np.nan
    curr_deg = deg[end_idx - 1] if end_idx > 0 else np.nan
    
    start_6h = np.searchsorted(times, t_np - np.timedelta64(6, "h"), side="left")
    wind_6h = wind[start_6h:end_idx]
    westerly_6h = westerly[start_6h:end_idx]
    
    start_24h = np.searchsorted(times, t_np - np.timedelta64(24, "h"), side="left")
    wind_24h = wind[start_24h:end_idx]
    westerly_24h = westerly[start_24h:end_idx]
    
    wind_mean_6 = np.mean(wind_6h) if len(wind_6h) > 0 else np.nan
    wind_max_6 = np.max(wind_6h) if len(wind_6h) > 0 else np.nan
    
    wind_mean_24 = np.mean(wind_24h) if len(wind_24h) > 0 else np.nan
    wind_max_24 = np.max(wind_24h) if len(wind_24h) > 0 else np.nan
    
    if len(wind_6h) > 0:
        max_idx_6h = np.argmax(wind_6h)
        west_max_6 = westerly_6h[max_idx_6h]
    else:
        west_max_6 = np.nan
        
    if len(wind_24h) > 0:
        max_idx_24h = np.argmax(wind_24h)
        west_max_24 = westerly_24h[max_idx_24h]
    else:
        west_max_24 = np.nan
        
    curr_wind = wind[end_idx - 1] if end_idx > 0 else np.nan
    westerly_strong_3 = int(curr_westerly == 1 and curr_wind >= 3.0) if pd.notna(curr_westerly) and pd.notna(curr_wind) else np.nan
    westerly_strong_5 = int(curr_westerly == 1 and curr_wind >= 5.0) if pd.notna(curr_westerly) and pd.notna(curr_wind) else np.nan
    
    results_s305.append({
        "analysis_id": row["analysis_id"],
        "wind_duration_3m_s": dur_3,
        "wind_duration_5m_s": dur_5,
        "wind_duration_local_p90": dur_p90,
        "westerly_current": curr_westerly,
        "풍향_sin_current": curr_sin,
        "풍향_cos_current": curr_cos,
        "wind_mean_6h": wind_mean_6,
        "wind_max_6h": wind_max_6,
        "wind_mean_24h": wind_mean_24,
        "wind_max_24h": wind_max_24,
        "westerly_strong_current_3m_s": westerly_strong_3,
        "westerly_strong_current_5m_s": westerly_strong_5,
        "westerly_strong_max_6h": west_max_6,
        "westerly_strong_max_24h": west_max_24,
        "시점_풍향_deg": curr_deg
    })

df_features_s305 = pd.DataFrame(results_s305)
merged_df_s305 = s3_05_input.merge(df_features_s305, on="analysis_id", how="left")

# Summarize continuous variables
def summarize_continuous_s305(df_wide, features):
    fire_part = df_wide[df_wide["target"] == 1].copy()
    control_part = df_wide[df_wide["target"] == 0].copy()
    
    control_means = control_part.groupby("fire_exposure_id", as_index=False)[features].mean()
    control_counts = control_part.groupby("fire_exposure_id", as_index=False).agg(control_row_n=("analysis_id", "count"))
    paired = fire_part.merge(control_means, on="fire_exposure_id", how="left", suffixes=("_fire", "_control_mean"))
    paired = paired.merge(control_counts, on="fire_exposure_id", how="left")
    
    delta_rows = []
    for f in features:
        fcol = f"{f}_fire"
        ccol = f"{f}_control_mean"
        tmp = paired[["fire_exposure_id", "기상셀ID", "기준시각", "기후지형유형", "date", "month", "hour", "matched_control_n", "full_5_matched", "is_ys0071", "is_less_than_5_matched", "is_2021_feb_cluster", "control_row_n", fcol, ccol]].copy()
        tmp = tmp.rename(columns={fcol: "fire_value", ccol: "control_mean"})
        tmp["feature"] = f
        tmp["delta"] = tmp["fire_value"] - tmp["control_mean"]
        tmp["risk_direction"] = "higher"
        tmp["risk_direction_match"] = tmp["delta"] > 0
        delta_rows.append(tmp)
        
    delta_long = pd.concat(delta_rows, ignore_index=True)
    
    groups = {
        "전체": delta_long.index.tolist(),
        "영동 해안형": delta_long[delta_long["기후지형유형"] == "영동 해안형"].index.tolist(),
        "영서 내륙형": delta_long[delta_long["기후지형유형"] == "영서 내륙형"].index.tolist(),
        "고지·산간형": delta_long[delta_long["기후지형유형"] == "고지·산간형"].index.tolist(),
        "영동 겨울": delta_long[(delta_long["기후지형유형"] == "영동 해안형") & (delta_long["month"].isin([12, 1, 2]))].index.tolist(),
        "영서/고지": delta_long[delta_long["기후지형유형"].isin(["영서 내륙형", "고지·산간형"])].index.tolist(),
    }
    
    stats_rows = []
    for grp_name, idxs in groups.items():
        sub = delta_long.loc[idxs]
        for feat, g in sub.groupby("feature", sort=False):
            complete = g[g["delta"].notna()].copy()
            x = complete["delta"].to_numpy(dtype=float)
            n = len(x)
            if n == 0: continue
            
            mean_delta = float(np.mean(x))
            sd_delta = float(np.std(x, ddof=1)) if n > 1 else np.nan
            dz = mean_delta / sd_delta if sd_delta and np.isfinite(sd_delta) and sd_delta != 0 else np.nan
            se = sd_delta / math.sqrt(n) if n > 1 and np.isfinite(sd_delta) else np.nan
            ci_low = mean_delta - stats.t.ppf(0.975, n - 1) * se if n > 1 and np.isfinite(se) else np.nan
            ci_high = mean_delta + stats.t.ppf(0.975, n - 1) * se if n > 1 and np.isfinite(se) else np.nan
            
            try: t_p = float(stats.ttest_1samp(x, 0, nan_policy="omit").pvalue) if n > 1 else np.nan
            except Exception: t_p = np.nan
            try:
                nonzero = x[x != 0]
                w_p = float(stats.wilcoxon(nonzero).pvalue) if len(nonzero) > 0 else np.nan
            except Exception: w_p = np.nan
            
            date_ci_low, date_ci_high = bootstrap_ci(x, complete["date"].to_numpy())
            cell_ci_low, cell_ci_high = bootstrap_ci(x, complete["기상셀ID"].to_numpy())
            
            direction_pct = float((x > 0).mean() * 100)
            
            stats_rows.append({
                "group": grp_name,
                "feature": feat,
                "n": int(n),
                "fire_mean": float(complete["fire_value"].mean()),
                "control_mean": float(complete["control_mean"].mean()),
                "mean_delta": mean_delta,
                "median_delta": float(np.median(x)),
                "sd_delta": sd_delta,
                "mean_delta_ci_low_t": ci_low,
                "mean_delta_ci_high_t": ci_high,
                "date_boot_ci_low": date_ci_low,
                "date_boot_ci_high": date_ci_high,
                "cell_boot_ci_low": cell_ci_low,
                "cell_boot_ci_high": cell_ci_high,
                "cohen_dz": dz,
                "direction_match_pct": direction_pct,
                "paired_t_p": t_p,
                "wilcoxon_p": w_p,
                "date_n": int(complete["date"].nunique()),
                "cell_n": int(complete["기상셀ID"].nunique()),
                "support_flag": "ok" if (n >= 30 and complete["date"].nunique() >= 10 and complete["기상셀ID"].nunique() >= 5) else "sparse_do_not_interpret"
            })
            
    summary_df = pd.DataFrame(stats_rows)
    summary_df["paired_t_q"] = bh_fdr(summary_df["paired_t_p"])
    summary_df["wilcoxon_q"] = bh_fdr(summary_df["wilcoxon_p"])
    return summary_df

features_to_sum_s305 = [
    "wind_duration_3m_s", "wind_duration_5m_s", "wind_duration_local_p90",
    "wind_mean_6h", "wind_max_6h", "wind_mean_24h", "wind_max_24h",
    "풍향_sin_current", "풍향_cos_current"
]

print("Summarizing wind duration & speed...")
wind_dur_summary = summarize_continuous_s305(merged_df_s305, features_to_sum_s305)
write_table(wind_dur_summary, "S3-05_wind_duration_summary.csv")

# Summarize binary variables
binary_specs_s305 = [
    ("westerly_current", "westerly_current", ">", 0.0),
    ("westerly_strong_current_3m_s", "westerly_strong_current_3m_s", ">", 0.0),
    ("westerly_strong_current_5m_s", "westerly_strong_current_5m_s", ">", 0.0),
    ("westerly_strong_max_6h", "westerly_strong_max_6h", ">", 0.0),
    ("westerly_strong_max_24h", "westerly_strong_max_24h", ">", 0.0),
]

def summarize_binary_s305(df_wide, specs):
    df_conds = df_wide[["analysis_id", "fire_exposure_id", "sample_group", "target", "기상셀ID", "기준시각", "기후지형유형", "date", "month", "hour"]].copy()
    for name, feat, op, val in specs:
        df_conds[name] = np.where(df_wide[feat].notna(), (df_wide[feat] > val).astype(float), np.nan)
        
    groups_def = {
        "전체": df_conds.index.tolist(),
        "영동 해안형": df_conds[df_conds["기후지형유형"] == "영동 해안형"].index.tolist(),
        "영서 내륙형": df_conds[df_conds["기후지형유형"] == "영서 내륙형"].index.tolist(),
        "고지·산간형": df_conds[df_conds["기후지형유형"] == "고지·산간형"].index.tolist(),
        "영동 겨울": df_conds[(df_conds["기후지형유형"] == "영동 해안형") & (df_conds["month"].isin([12, 1, 2]))].index.tolist(),
        "영서/고지": df_conds[df_conds["기후지형유형"].isin(["영서 내륙형", "고지·산간형"])].index.tolist(),
    }
    
    cond_rows = []
    for grp_name, idxs in groups_def.items():
        sub_df = df_conds.loc[idxs]
        for spec in specs:
            cond = spec[0]
            
            fire = sub_df[sub_df["target"] == 1][["fire_exposure_id", "기상셀ID", "date", cond]].copy()
            ctrl = sub_df[sub_df["target"] == 0][["fire_exposure_id", cond]].copy()
            
            ctrl_mean = ctrl.groupby("fire_exposure_id", as_index=False)[cond].mean().rename(columns={cond: "control_rate"})
            paired = fire.merge(ctrl_mean, on="fire_exposure_id", how="left")
            paired = paired.dropna(subset=[cond, "control_rate"])
            
            diff = paired[cond] - paired["control_rate"]
            n = len(paired)
            if n == 0: continue
            
            fire_rate = float(paired[cond].mean())
            control_rate = float(paired["control_rate"].mean())
            rd = float(diff.mean())
            lo, hi = bootstrap_ci(diff.to_numpy(), paired["date"].to_numpy())
            
            try: p = float(stats.ttest_1samp(diff, 0, nan_policy="omit").pvalue) if n > 1 else np.nan
            except Exception: p = np.nan
            
            pr = (fire_rate + 1e-9) / (control_rate + 1e-9)
            odds_fire = (fire_rate + 0.5 / n) / (1.0 - fire_rate + 0.5 / n)
            odds_control = (control_rate + 0.5 / n) / (1.0 - control_rate + 0.5 / n)
            or_val = odds_fire / odds_control if odds_control else np.nan
            
            cond_rows.append({
                "group": grp_name,
                "condition": cond,
                "n": int(n),
                "fire_condition_rate": fire_rate,
                "control_condition_rate": control_rate,
                "matched_risk_diff": rd,
                "date_boot_ci_low": lo,
                "date_boot_ci_high": hi,
                "matched_prevalence_ratio": pr,
                "matched_or_approx": or_val,
                "paired_t_p": p,
                "date_n": int(paired["date"].nunique()),
                "cell_n": int(paired["기상셀ID"].nunique()),
                "support_flag": "ok" if (n >= 30 and paired["date"].nunique() >= 10 and paired["기상셀ID"].nunique() >= 5) else "sparse_do_not_interpret"
            })
            
    summary_df = pd.DataFrame(cond_rows)
    summary_df["paired_t_q"] = bh_fdr(summary_df["paired_t_p"])
    return summary_df

print("Summarizing wind direction binary conditions...")
wind_dir_cond_summary = summarize_binary_s305(merged_df_s305, binary_specs_s305)
write_table(wind_dir_cond_summary, "S3-05_wind_direction_condition_summary.csv")

# Circular stats
def compute_circular_stats_s305(df_wide):
    groups_def = {
        "전체": df_wide.index.tolist(),
        "영동 해안형": df_wide[df_wide["기후지형유형"] == "영동 해안형"].index.tolist(),
        "영서 내륙형": df_wide[df_wide["기후지형유형"] == "영서 내륙형"].index.tolist(),
        "고지·산간형": df_wide[df_wide["기후지형유형"] == "고지·산간형"].index.tolist(),
        "영동 겨울": df_wide[(df_wide["기후지형유형"] == "영동 해안형") & (df_wide["month"].isin([12, 1, 2]))].index.tolist(),
        "영서/고지": df_wide[df_wide["기후지형유형"].isin(["영서 내륙형", "고지·산간형"])].index.tolist(),
    }
    
    circular_rows = []
    for grp_name, idxs in groups_def.items():
        sub = df_wide.loc[idxs]
        for target_val, target_name in [(1, "fire"), (0, "control")]:
            sub_t = sub[sub["target"] == target_val].copy().dropna(subset=["시점_풍향_deg"])
            n = len(sub_t)
            if n == 0: continue
            
            sin_vals = sub_t["풍향_sin_current"].to_numpy()
            cos_vals = sub_t["풍향_cos_current"].to_numpy()
            
            mean_sin = float(np.mean(sin_vals))
            mean_cos = float(np.mean(cos_vals))
            
            R = float(np.sqrt(mean_sin**2 + mean_cos**2))
            
            mean_angle = float(np.degrees(np.arctan2(mean_sin, mean_cos)))
            if mean_angle < 0:
                mean_angle += 360.0
                
            z = n * (R**2)
            rayleigh_p = float(np.exp(-z))
            
            circular_rows.append({
                "group": grp_name,
                "sample_group": target_name,
                "n": int(n),
                "mean_sin": mean_sin,
                "mean_cos": mean_cos,
                "R_value": R,
                "mean_angle_deg": mean_angle,
                "rayleigh_z": z,
                "rayleigh_p": rayleigh_p
            })
            
    return pd.DataFrame(circular_rows)

print("Computing circular stats...")
circular_stats_df = compute_circular_stats_s305(merged_df_s305)
write_table(circular_stats_df, "S3-05_circular_direction_tests.csv")
print("S3-05 calculations completed!")


In [27]:
# S3-05 플롯: Wind Rose 플롯과 효과크기 forest plot을 저장한다.
print("Creating S3-05 Wind Rose plot...")
plt.figure(figsize=(7, 7))
ax = plt.subplot(111, polar=True)

# Filter for Yeongdong Coastal Type
yd_df = merged_df_s305[merged_df_s305["기후지형유형"] == "영동 해안형"].dropna(subset=["시점_풍향_deg"])
yd_fire = yd_df[yd_df["target"] == 1]["시점_풍향_deg"].to_numpy()
yd_ctrl = yd_df[yd_df["target"] == 0]["시점_풍향_deg"].to_numpy()

bins = np.linspace(0, 2*np.pi, 17)
fire_rad = np.radians(yd_fire)
ctrl_rad = np.radians(yd_ctrl)

fire_counts, _ = np.histogram(fire_rad, bins=bins)
ctrl_counts, _ = np.histogram(ctrl_rad, bins=bins)

fire_rates = fire_counts / len(yd_fire)
ctrl_rates = ctrl_counts / len(yd_ctrl)

centers = (bins[:-1] + bins[1:]) / 2
width = 2 * np.pi / 16

ax.bar(centers, ctrl_rates, width=width, color="#4c78a8", alpha=0.5, edgecolor="#4c78a8", label="Matched Control")
ax.bar(centers, fire_rates, width=width, color="#d95f02", alpha=0.5, edgecolor="#d95f02", label="Fire")

ax.set_theta_zero_location("N")
ax.set_theta_direction(-1) # clockwise
ax.set_xticklabels(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'])

plt.legend(loc="lower right", bbox_to_anchor=(1.1, 0), frameon=True)
plt.title("S3-05 Wind Rose: Yeongdong Coastal Type Wind Direction Frequency", pad=15)
wind_rose_path = save_current_plot("S3-05_wind_rose_by_type.png")

print("Creating S3-05 forest plot...")
forest_feats = [
    "wind_mean_6h", "wind_max_6h", "wind_duration_3m_s", "wind_duration_5m_s",
    "풍향_sin_current", "풍향_cos_current"
]
forest_df = wind_dur_summary[
    (wind_dur_summary["feature"].isin(forest_feats)) &
    (wind_dur_summary["group"].isin(["전체", "영동 해안형", "영동 겨울"]))
].copy()

forest_df["label"] = forest_df["group"] + " | " + forest_df["feature"]
forest_df = forest_df.sort_values("cohen_dz")

plt.figure(figsize=(9, 6))
y = np.arange(len(forest_df))
scaled_low = (forest_df["mean_delta"] - forest_df["date_boot_ci_low"]).abs() / forest_df["sd_delta"].replace(0, np.nan)
scaled_high = (forest_df["date_boot_ci_high"] - forest_df["mean_delta"]).abs() / forest_df["sd_delta"].replace(0, np.nan)

plt.errorbar(forest_df["cohen_dz"], y, xerr=[scaled_low, scaled_high], fmt="o", color="#333333", ecolor="#888888", capsize=3, label="Cohen's dz (date-bootstrap CI scaled)")
plt.axvline(0, color="#666666", linewidth=1, linestyle="--")
plt.yticks(y, forest_df["label"])
plt.xlabel("Cohen's dz (Effect Size)")
plt.title("S3-05 Wind Speed, Duration and Direction delta Effect Sizes")
forest_plot_path = save_current_plot("S3-05_wind_threshold_forest.png")

plot_paths_s305 = [wind_rose_path, forest_plot_path]
plot_manifest_s305 = pd.DataFrame([{"plot_path": str(p.relative_to(REPO_ROOT)), "file_name": p.name} for p in plot_paths_s305])
write_table(plot_manifest_s305, "S3-05_plot_manifest.csv")

artifact_manifest_s305 = pd.DataFrame([
    { "artifact_type": "table", "path": str((TABLE_DIR / name).relative_to(REPO_ROOT)).replace("\\", "/") }
    for name in ["S3-05_wind_duration_summary.csv", "S3-05_wind_direction_condition_summary.csv", "S3-05_circular_direction_tests.csv", "S3-05_plot_manifest.csv"]
] + [
    {"artifact_type": "plot", "path": str(p.relative_to(REPO_ROOT)).replace("\\", "/")}
    for p in plot_paths_s305
])
write_table(artifact_manifest_s305, "S3-05_artifact_manifest.csv")
print("S3-05 plots and artifact manifest completed!")


## S3-06 : 기온·기압 이상성 보조 감사

Step 2에서 매칭 후 기온·기압 평균 차이가 거의 사라진 결과를 기준으로, 특정 층화에서만 이상성이 남는지 보조 확인한다. 한랭/온난, 기압 급변, 일교차 확대를 원인 후보가 아니라 교란·프록시 감사로 다룬다.

### S3-06-1. 분석 범위 및 파생 변수 생성

In [30]:
# S3-06 분석 범위 고정 및 감사 데이터셋 구성
S3_ID = "S3-06"
S3_06_SCOPE = pd.DataFrame([
    {"item": "analysis_question", "value": "Step 2에서 매칭 후 기온·기압 평균 차이가 사라진 기후 배경이 특정 층화나 이상성 관점에서 남는가?"},
    {"item": "analysis_unit", "value": "fire_exposure_id별 대응 delta"},
    {"item": "population", "value": "Step 2 고유 산불 노출 1,150개"},
    {"item": "comparison_group", "value": "Step 2 고정 매칭 대조군 5,632행"},
    {"item": "period", "value": "2020~2021년 강원도 기상셀 시간 기상자료"},
    {"item": "control_stratification_matching", "value": "동일 기상셀, 동일 연도, 동일 월, 동일 hour 매칭"},
])
write_table(S3_06_SCOPE, "S3-06_analysis_scope.csv")

if "analysis_keys" not in globals():
    raise RuntimeError("S3-01 analysis_keys가 필요합니다. 앞 셀을 먼저 실행하십시오.")
if analysis_keys["analysis_id"].duplicated().any():
    raise ValueError("analysis_id duplicated before S3-06")

s3_06_input = analysis_keys.copy()
s3_06_input["기준시각"] = pd.to_datetime(s3_06_input["기준시각"])
s3_06_input["date"] = s3_06_input["기준시각"].dt.date.astype(str)

print("Loading weather data for S3-06...")
raw_weather_path = DATA_DIR / "기상_시간단위_파생.csv"
raw_weather_head = pd.read_csv(raw_weather_path, encoding="utf-8-sig", nrows=5)

cell_col = [c for c in raw_weather_head.columns if "셀ID" in c][0]
time_col = [c for c in raw_weather_head.columns if "일시" in c][0]
temp_col = [c for c in raw_weather_head.columns if "기온_C" in c][0]
press_col = [c for c in raw_weather_head.columns if "현지기압" in c][0]
press_diff_col = [c for c in raw_weather_head.columns if "기압변동_3h" in c][0]
humid_col = [c for c in raw_weather_head.columns if "습도_pct" in c][0]
wind_col = [c for c in raw_weather_head.columns if "풍속_m_s" in c][0]

required_cols = [cell_col, time_col, temp_col, press_col, press_diff_col, humid_col, wind_col]
hourly_weather_s306 = pd.read_csv(raw_weather_path, usecols=required_cols, encoding="utf-8-sig", parse_dates=[time_col])
hourly_weather_s306 = hourly_weather_s306.sort_values([cell_col, time_col]).reset_index(drop=True)

hourly_weather_s306["month"] = hourly_weather_s306[time_col].dt.month
hourly_weather_s306["hour"] = hourly_weather_s306[time_col].dt.hour
hourly_weather_s306["temp"] = hourly_weather_s306[temp_col]
hourly_weather_s306["press"] = hourly_weather_s306[press_col]

print("Calculating rolling window features for S3-06...")
hourly_weather_s306["dtr"] = hourly_weather_s306.groupby(cell_col)["temp"].transform(
    lambda x: x.shift(1).rolling(24, min_periods=24).max() - x.shift(1).rolling(24, min_periods=24).min()
)
hourly_weather_s306["press_diff"] = hourly_weather_s306.groupby(cell_col)[press_diff_col].transform(
    lambda x: x.shift(1)
)

# 결합분석용 변수 (습도 및 풍속 3h 급변)
hourly_weather_s306["humid_diff_3h"] = hourly_weather_s306.groupby(cell_col)[humid_col].transform(
    lambda x: x.shift(1) - x.shift(4)
)
hourly_weather_s306["wind_diff_3h"] = hourly_weather_s306.groupby(cell_col)[wind_col].transform(
    lambda x: x.shift(1) - x.shift(4)
)

print("Calculating local z-scores and percentile ranks...")
group_cols = [cell_col, "month", "hour"]
feature_cols = ["temp", "press", "dtr", "press_diff"]

for col in feature_cols:
    mean_val = hourly_weather_s306.groupby(group_cols)[col].transform("mean")
    std_val = hourly_weather_s306.groupby(group_cols)[col].transform("std")
    std_val = std_val.replace(0, np.nan)
    hourly_weather_s306[f"{col}_z"] = (hourly_weather_s306[col] - mean_val) / std_val
    hourly_weather_s306[f"{col}_pct"] = hourly_weather_s306.groupby(group_cols)[col].rank(pct=True)

check_cols = []
for f in feature_cols:
    check_cols.extend([f, f"{f}_z", f"{f}_pct"])
joint_cols = ["humid_diff_3h", "wind_diff_3h"]

print("Merging with analysis_keys...")
ak_cell_col = [c for c in s3_06_input.columns if "셀ID" in c][0]
ak_time_col = [c for c in s3_06_input.columns if "기준시각" in c][0]

merged_df_s306 = s3_06_input.merge(
    hourly_weather_s306[[cell_col, time_col] + check_cols + joint_cols],
    left_on=[ak_cell_col, ak_time_col],
    right_on=[cell_col, time_col],
    how="left"
)
print("Merged df shape:", merged_df_s306.shape)


### S3-06-2. 통계 요약 및 결합 분석

In [32]:
# delta_raw 구축 및 통계 집계
print("Generating delta_raw...")
fire_df = merged_df_s306[merged_df_s306["target"] == 1].copy()
control_df = merged_df_s306[merged_df_s306["target"] == 0].copy()
control_means = control_df.groupby("fire_exposure_id")[check_cols + joint_cols].mean()

delta_df = fire_df.merge(
    control_means,
    on="fire_exposure_id",
    suffixes=("_fire", "_control_mean")
)

delta_rows = []
for f in check_cols + joint_cols:
    sub = delta_df[["fire_exposure_id", ak_cell_col, "date", "month", "hour", "기후지형유형", f"{f}_fire", f"{f}_control_mean"]].copy()
    sub["feature"] = f
    sub["fire_value"] = sub[f"{f}_fire"]
    sub["control_mean"] = sub[f"{f}_control_mean"]
    sub["delta"] = sub["fire_value"] - sub["control_mean"]
    delta_rows.append(sub[["fire_exposure_id", ak_cell_col, "date", "month", "hour", "기후지형유형", "feature", "fire_value", "control_mean", "delta"]])

delta_raw_s306 = pd.concat(delta_rows, ignore_index=True)

BOOT_N_S306 = 500
rng_s306 = np.random.default_rng(306)

def bootstrap_ci_s306(values, blocks, reps=BOOT_N_S306):
    vals = np.asarray(values, dtype=float)
    block_arr = np.asarray(blocks)
    ok = np.isfinite(vals) & pd.notna(block_arr)
    vals = vals[ok]
    block_arr = block_arr[ok]
    if len(vals) < 2: return np.nan, np.nan
    unique_blocks = pd.unique(block_arr)
    if len(unique_blocks) < 2: return np.nan, np.nan
    by_block = {b: vals[block_arr == b] for b in unique_blocks}
    means = np.empty(reps)
    for r in range(reps):
        sample_blocks = rng_s306.choice(unique_blocks, size=len(unique_blocks), replace=True)
        sample_vals = np.concatenate([by_block[b] for b in sample_blocks])
        means[r] = np.nanmean(sample_vals)
    return float(np.nanpercentile(means, 2.5)), float(np.nanpercentile(means, 97.5))

feature_meta_s306 = pd.DataFrame([
    {"feature": "temp", "variable": "Temperature", "feature_family": "Temperature", "window_h": 0, "risk_direction": "higher"},
    {"feature": "temp_z", "variable": "Temperature (Z)", "feature_family": "Temperature Anomaly", "window_h": 0, "risk_direction": "higher"},
    {"feature": "temp_pct", "variable": "Temperature (Percentile)", "feature_family": "Temperature Anomaly", "window_h": 0, "risk_direction": "higher"},
    {"feature": "press", "variable": "Pressure", "feature_family": "Pressure", "window_h": 0, "risk_direction": "lower"},
    {"feature": "press_z", "variable": "Pressure (Z)", "feature_family": "Pressure Anomaly", "window_h": 0, "risk_direction": "lower"},
    {"feature": "press_pct", "variable": "Pressure (Percentile)", "feature_family": "Pressure Anomaly", "window_h": 0, "risk_direction": "lower"},
    {"feature": "dtr", "variable": "Diurnal Temp Range", "feature_family": "Temperature Range", "window_h": 24, "risk_direction": "higher"},
    {"feature": "dtr_z", "variable": "Diurnal Temp Range (Z)", "feature_family": "Temperature Range Anomaly", "window_h": 24, "risk_direction": "higher"},
    {"feature": "dtr_pct", "variable": "Diurnal Temp Range (Percentile)", "feature_family": "Temperature Range Anomaly", "window_h": 24, "risk_direction": "higher"},
    {"feature": "press_diff", "variable": "Pressure Change 3h", "feature_family": "Pressure Change", "window_h": 3, "risk_direction": "lower"},
    {"feature": "press_diff_z", "variable": "Pressure Change 3h (Z)", "feature_family": "Pressure Change Anomaly", "window_h": 3, "risk_direction": "lower"},
    {"feature": "press_diff_pct", "variable": "Pressure Change 3h (Percentile)", "feature_family": "Pressure Change Anomaly", "window_h": 3, "risk_direction": "lower"},
])

def summarize_delta_s306(sub, group_label):
    rows = []
    meta_lookup = feature_meta_s306.set_index("feature")
    sub_filtered = sub[sub["feature"].isin(feature_meta_s306["feature"])].copy()
    for feature, g in sub_filtered.groupby("feature", sort=False):
        meta = meta_lookup.loc[feature]
        complete = g[g["delta"].notna()].copy()
        x = complete["delta"].to_numpy(dtype=float)
        n = len(x)
        if n == 0: continue
        mean_delta = float(np.mean(x))
        sd_delta = float(np.std(x, ddof=1)) if n > 1 else np.nan
        dz = mean_delta / sd_delta if sd_delta and np.isfinite(sd_delta) and sd_delta != 0 else np.nan
        se = sd_delta / math.sqrt(n) if n > 1 and np.isfinite(sd_delta) else np.nan
        ci_low = mean_delta - stats.t.ppf(0.975, n - 1) * se if n > 1 and np.isfinite(se) else np.nan
        ci_high = mean_delta + stats.t.ppf(0.975, n - 1) * se if n > 1 and np.isfinite(se) else np.nan
        try: t_p = float(stats.ttest_1samp(x, 0, nan_policy="omit").pvalue) if n > 1 else np.nan
        except Exception: t_p = np.nan
        try:
            nonzero = x[x != 0]
            w_p = float(stats.wilcoxon(nonzero).pvalue) if len(nonzero) > 0 else np.nan
        except Exception: w_p = np.nan
        date_ci_low, date_ci_high = bootstrap_ci_s306(x, complete["date"].to_numpy())
        cell_ci_low, cell_ci_high = bootstrap_ci_s306(x, complete[ak_cell_col].to_numpy())
        risk_direction = meta["risk_direction"]
        direction_pct = float(((x < 0) if risk_direction == "lower" else (x > 0)).mean() * 100)
        rows.append({
            "group": group_label,
            "feature": feature,
            "variable": meta["variable"],
            "feature_family": meta["feature_family"],
            "window_h": meta["window_h"],
            "risk_direction": risk_direction,
            "n": int(n),
            "fire_mean": float(complete["fire_value"].mean()),
            "control_mean": float(complete["control_mean"].mean()),
            "mean_delta": mean_delta,
            "median_delta": float(np.median(x)),
            "sd_delta": sd_delta,
            "mean_delta_ci_low_t": ci_low,
            "mean_delta_ci_high_t": ci_high,
            "date_boot_ci_low": date_ci_low,
            "date_boot_ci_high": date_ci_high,
            "cell_boot_ci_low": cell_ci_low,
            "cell_boot_ci_high": cell_ci_high,
            "cohen_dz": dz,
            "direction_match_pct": direction_pct,
            "paired_t_p": t_p,
            "wilcoxon_p": w_p,
            "date_n": int(complete["date"].nunique()),
            "cell_n": int(complete[ak_cell_col].nunique()),
            "support_flag": "ok" if (n >= 30 and complete["date"].nunique() >= 10 and complete[ak_cell_col].nunique() >= 5) else "sparse_do_not_interpret"
        })
    return pd.DataFrame(rows)

print("Summarizing delta...")
summary_parts = [summarize_delta_s306(delta_raw_s306, "전체")]
for climate_type, sub in delta_raw_s306.groupby("기후지형유형", sort=False):
    summary_parts.append(summarize_delta_s306(sub, climate_type))
delta_summary_s306 = pd.concat(summary_parts, ignore_index=True)
delta_summary_s306["paired_t_q"] = bh_fdr(delta_summary_s306["paired_t_p"])
delta_summary_s306["wilcoxon_q"] = bh_fdr(delta_summary_s306["wilcoxon_p"])
delta_summary_s306["date_ci_excludes_zero"] = (delta_summary_s306["date_boot_ci_low"] > 0) | (delta_summary_s306["date_boot_ci_high"] < 0)
delta_summary_s306["cell_ci_excludes_zero"] = (delta_summary_s306["cell_boot_ci_low"] > 0) | (delta_summary_s306["cell_boot_ci_high"] < 0)

write_table(delta_summary_s306, "S3-06_temperature_pressure_anomaly_summary.csv")

# CSV 2: 월별 x 기후지형유형별 z-score delta 평균
print("Generating anomaly_by_type_month...")
z_features = ["temp_z", "press_z", "dtr_z", "press_diff_z"]
sub_z = delta_raw_s306[delta_raw_s306["feature"].isin(z_features)].copy()
anomaly_by_type_month = sub_z.groupby(["month", "기후지형유형", "feature"])["delta"].mean().unstack("feature").reset_index()
write_table(anomaly_by_type_month, "S3-06_anomaly_by_type_month.csv")

# CSV 3: 기압 급변 x 기상 급변 결합분석
print("Generating joint summary...")
merged_df_s306["press_drop"] = merged_df_s306["press_diff_z"] <= -1.0
merged_df_s306["humid_drop"] = merged_df_s306["humid_diff_3h"] <= -5.0
merged_df_s306["wind_rise"] = merged_df_s306["wind_diff_3h"] >= 1.5

joint_rows = []
groups_dict = {"전체": merged_df_s306.copy()}
for ct, sub in merged_df_s306.groupby("기후지형유형"):
    groups_dict[ct] = sub.copy()

for grp_name, df_g in groups_dict.items():
    n_fire = len(df_g[df_g["target"] == 1])
    n_control = len(df_g[df_g["target"] == 0])
    if n_fire == 0 or n_control == 0: continue
    
    conditions = {
        "press_drop_only": df_g["press_drop"],
        "humid_drop_only": df_g["humid_drop"],
        "wind_rise_only": df_g["wind_rise"],
        "press_drop_AND_humid_drop": df_g["press_drop"] & df_g["humid_drop"],
        "press_drop_AND_wind_rise": df_g["press_drop"] & df_g["wind_rise"],
        "humid_drop_AND_wind_rise": df_g["humid_drop"] & df_g["wind_rise"],
        "press_drop_AND_humid_drop_AND_wind_rise": df_g["press_drop"] & df_g["humid_drop"] & df_g["wind_rise"]
    }
    
    for cond_name, cond_series in conditions.items():
        fire_ok = df_g[df_g["target"] == 1][cond_name] if cond_name in df_g.columns else cond_series[df_g["target"] == 1]
        control_ok = df_g[df_g["target"] == 0][cond_name] if cond_name in df_g.columns else cond_series[df_g["target"] == 0]
        
        rate_fire = float(fire_ok.mean())
        rate_control = float(control_ok.mean())
        diff_rate = rate_fire - rate_control
        
        odds_fire = (rate_fire + 0.5 / n_fire) / (1.0 - rate_fire + 0.5 / n_fire)
        odds_control = (rate_control + 0.5 / n_control) / (1.0 - rate_control + 0.5 / n_control)
        or_val = odds_fire / odds_control if odds_control else np.nan
        
        joint_rows.append({
            "group": grp_name,
            "condition": cond_name,
            "fire_n": n_fire,
            "control_n": n_control,
            "fire_rate": rate_fire,
            "control_rate": rate_control,
            "delta_rate": diff_rate,
            "approx_or": or_val
        })

joint_summary_s306 = pd.DataFrame(joint_rows)
write_table(joint_summary_s306, "S3-06_pressure_change_joint_summary.csv")
print("S3-06 statistics calculation completed!")

### S3-06-3. 핵심 플롯 저장 및 최종 감사

In [34]:
# S3-06 시각화 및 감사
print("Creating S3-06 anomaly delta heatmaps...")
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

temp_pivot = anomaly_by_type_month.pivot(index="month", columns="기후지형유형", values="temp_z")
sns.heatmap(temp_pivot, annot=True, fmt=".3f", cmap="RdBu_r", center=0, ax=axes[0])
axes[0].set_title("월별 × 기후유형별 기온 Anomaly Delta 평균 (Z-score)")
axes[0].set_xlabel("기후지형유형")
axes[0].set_ylabel("월")

press_pivot = anomaly_by_type_month.pivot(index="month", columns="기후지형유형", values="press_z")
sns.heatmap(press_pivot, annot=True, fmt=".3f", cmap="RdBu_r", center=0, ax=axes[1])
axes[1].set_title("월별 × 기후유형별 기압 Anomaly Delta 평균 (Z-score)")
axes[1].set_xlabel("기후지형유형")
axes[1].set_ylabel("월")

heatmap_path = save_current_plot("S3-06_anomaly_heatmap.png")

print("Creating S3-06 pressure change distribution...")
plt.figure(figsize=(10, 6))
plot_data = merged_df_s306.dropna(subset=["press_diff_z"]).copy()
group_mapping = {"fire": "산불 노출 (Fire)", "matched_control": "매칭 대조군 (Control)"}
plot_data["집단"] = plot_data["sample_group"].map(group_mapping)

sns.kdeplot(
    data=plot_data,
    x="press_diff_z",
    hue="집단",
    fill=True,
    common_norm=False,
    palette={"산불 노출 (Fire)": "#d95f02", "매칭 대조군 (Control)": "#4c78a8"},
    alpha=0.4,
    linewidth=2
)
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.title("산불 노출 vs 매칭 대조군 기압변동 Z-score (press_diff_z) 분포 비교")
plt.xlabel("기압변동 Z-score (3h 전 대비, t-1 시점)")
plt.ylabel("밀도 (Density)")
plt.xlim(-4, 4)

dist_path = save_current_plot("S3-06_pressure_change_distribution.png")

# 매니페스트 저장
plot_paths = [heatmap_path, dist_path]
plot_manifest_s306 = pd.DataFrame([{"plot_path": str(p.relative_to(REPO_ROOT)).replace("\\", "/"), "file_name": p.name} for p in plot_paths])
write_table(plot_manifest_s306, "S3-06_plot_manifest.csv")

artifact_manifest_s306 = pd.DataFrame([
    { "artifact_type": "table", "path": f"jsw/강원_재_EDA/outputs/Step3/tables/{name}" }
    for name in ["S3-06_temperature_pressure_anomaly_summary.csv", "S3-06_anomaly_by_type_month.csv", "S3-06_pressure_change_joint_summary.csv", "S3-06_plot_manifest.csv"]
] + [
    {"artifact_type": "plot", "path": f"jsw/강원_재_EDA/outputs/Step3/plots/{p.name}"}
    for p in plot_paths
])
write_table(artifact_manifest_s306, "S3-06_artifact_manifest.csv")
print("S3-06 plots and artifact manifest completed!")
delta_summary_s306[delta_summary_s306["group"].eq("전체")].sort_values("cohen_dz", key=lambda s: s.abs(), ascending=False)


## S3-07 : 절대습도와 국지 상대건조 분위수

절대습도(AH)와 포화수증기압 격차(VPD)를 계산하여 기후 배경에 따른 건조 신호의 판별 성능을 분석하고, 공통 절대 임계치 기준과 기상셀×월×hour 국지 상대 분위수 임계치 기준의 안정성을 비교 검증한다.

### S3-07-1. 절대습도 및 VPD 파생 변수 생성

In [37]:
# S3-07 분석 범위 정의 및 데이터 파생
S3_ID = "S3-07"
S3_07_SCOPE = pd.DataFrame([
    {"item": "analysis_question", "value": "절대적 건조값 기준보다 셀x월xhour 국지 상대 백분위(분위수) 기준이 지형/계절적 기온 편차를 제어하여 더 안정적으로 산불 위험을 포착하는가?"},
    {"item": "analysis_unit", "value": "fire_exposure_id별 대응 delta 및 이진 조건 충족률 격차"},
    {"item": "population", "value": "Step 2 고유 산불 노출 1,150개"},
    {"item": "comparison_group", "value": "Step 2 고정 매칭 대조군 5,632행"},
    {"item": "period", "value": "2020~2021년 강원도 기상셀 시간 기상자료"},
    {"item": "control_stratification_matching", "value": "동일 기상셀, 동일 연도, 동일 월, 동일 hour 매칭"},
])
write_table(S3_07_SCOPE, "S3-07_analysis_scope.csv")

if "analysis_keys" not in globals():
    raise RuntimeError("S3-01 analysis_keys가 필요합니다. 앞 셀을 먼저 실행하십시오.")
if analysis_keys["analysis_id"].duplicated().any():
    raise ValueError("analysis_id duplicated before S3-07")

s3_07_input = analysis_keys.copy()
s3_07_input["기준시각"] = pd.to_datetime(s3_07_input["기준시각"])
s3_07_input["date"] = s3_07_input["기준시각"].dt.date.astype(str)

print("Loading weather data for S3-07...")
raw_weather_path = DATA_DIR / "기상_시간단위_파생.csv"
raw_weather_head = pd.read_csv(raw_weather_path, encoding="utf-8-sig", nrows=5)

cell_col = [c for c in raw_weather_head.columns if "셀ID" in c][0]
time_col = [c for c in raw_weather_head.columns if "일시" in c][0]
temp_col = [c for c in raw_weather_head.columns if "기온_C" in c][0]
humid_col = [c for c in raw_weather_head.columns if "습도_pct" in c][0]

required_cols = [cell_col, time_col, temp_col, humid_col]
hourly_weather_s307 = pd.read_csv(raw_weather_path, usecols=required_cols, encoding="utf-8-sig", parse_dates=[time_col])
hourly_weather_s307 = hourly_weather_s307.sort_values([cell_col, time_col]).reset_index(drop=True)

print("Calculating Es, Ea, VPD, AH...")
temp_arr = hourly_weather_s307[temp_col].to_numpy()
rh_arr = hourly_weather_s307[humid_col].to_numpy()
es_arr = 0.61078 * np.exp((17.27 * temp_arr) / (temp_arr + 237.3))
ea_arr = es_arr * (rh_arr / 100.0)

hourly_weather_s307["temp"] = temp_arr
hourly_weather_s307["rh"] = rh_arr
hourly_weather_s307["vpd"] = es_arr - ea_arr
hourly_weather_s307["ah"] = 216.7 * ea_arr / (temp_arr + 273.15)

hourly_weather_s307["month"] = hourly_weather_s307[time_col].dt.month
hourly_weather_s307["hour"] = hourly_weather_s307[time_col].dt.hour

print("Calculating group quantiles...")
group_cols = [cell_col, "month", "hour"]
grouped_s307 = hourly_weather_s307.groupby(group_cols)

quantiles_rh = grouped_s307["rh"].quantile([0.05, 0.10, 0.20]).unstack().reset_index()
quantiles_rh.columns = group_cols + ["rh_q05", "rh_q10", "rh_q20"]

quantiles_ah = grouped_s307["ah"].quantile([0.05, 0.10, 0.20]).unstack().reset_index()
quantiles_ah.columns = group_cols + ["ah_q05", "ah_q10", "ah_q20"]

quantiles_vpd = grouped_s307["vpd"].quantile([0.80, 0.90, 0.95]).unstack().reset_index()
quantiles_vpd.columns = group_cols + ["vpd_q80", "vpd_q90", "vpd_q95"]

print("Merging quantiles back to hourly_weather...")
hourly_weather_s307 = hourly_weather_s307.merge(quantiles_rh, on=group_cols, how="left")
hourly_weather_s307 = hourly_weather_s307.merge(quantiles_ah, on=group_cols, how="left")
hourly_weather_s307 = hourly_weather_s307.merge(quantiles_vpd, on=group_cols, how="left")

print("Calculating local z-scores and percentiles...")
for col in ["rh", "ah", "vpd"]:
    mean_val = hourly_weather_s307.groupby(group_cols)[col].transform("mean")
    std_val = hourly_weather_s307.groupby(group_cols)[col].transform("std")
    std_val = std_val.replace(0, np.nan)
    hourly_weather_s307[f"{col}_z"] = (hourly_weather_s307[col] - mean_val) / std_val
    hourly_weather_s307[f"{col}_pct"] = hourly_weather_s307.groupby(group_cols)[col].rank(pct=True)

temp_save_cols = [
    cell_col, time_col, "month", "hour", "temp", "rh", "ah", "vpd",
    "rh_z", "rh_pct", "ah_z", "ah_pct", "vpd_z", "vpd_pct",
    "rh_q05", "rh_q10", "rh_q20", "ah_q05", "ah_q10", "ah_q20", "vpd_q80", "vpd_q90", "vpd_q95"
]
write_table(hourly_weather_s307[temp_save_cols], "S3-07_hourly_weather_temp.csv.gz", compression="gzip")

print("Merging with analysis_keys...")
ak_cell_col = [c for c in s3_07_input.columns if "셀ID" in c][0]
ak_time_col = [c for c in s3_07_input.columns if "기준시각" in c][0]
hw_merge_cols = [c for c in temp_save_cols if c not in ["month", "hour"] or c in [cell_col, time_col]]

merged_df_s307 = s3_07_input.merge(
    hourly_weather_s307[hw_merge_cols],
    left_on=[ak_cell_col, ak_time_col],
    right_on=[cell_col, time_col],
    how="left"
)
print("Merged df shape:", merged_df_s307.shape)


### S3-07-2. 분위수 임계치 요약 및 통계적 검증

In [39]:
# 이진 플래그 생성 및 통계 요약
conditions_def = {
    "rh_le_20": merged_df_s307["rh"] <= 20,
    "rh_le_30": merged_df_s307["rh"] <= 30,
    "rh_le_40": merged_df_s307["rh"] <= 40,
    "ah_le_2": merged_df_s307["ah"] <= 2.0,
    "ah_le_3": merged_df_s307["ah"] <= 3.0,
    "ah_le_4": merged_df_s307["ah"] <= 4.0,
    "vpd_ge_0p5": merged_df_s307["vpd"] >= 0.5,
    "vpd_ge_1p0": merged_df_s307["vpd"] >= 1.0,
    "vpd_ge_1p5": merged_df_s307["vpd"] >= 1.5,
    "rh_local_q05": merged_df_s307["rh"] <= merged_df_s307["rh_q05"],
    "rh_local_q10": merged_df_s307["rh"] <= merged_df_s307["rh_q10"],
    "rh_local_q20": merged_df_s307["rh"] <= merged_df_s307["rh_q20"],
    "ah_local_q05": merged_df_s307["ah"] <= merged_df_s307["ah_q05"],
    "ah_local_q10": merged_df_s307["ah"] <= merged_df_s307["ah_q10"],
    "ah_local_q20": merged_df_s307["ah"] <= merged_df_s307["ah_q20"],
    "vpd_local_q80": merged_df_s307["vpd"] >= merged_df_s307["vpd_q80"],
    "vpd_local_q90": merged_df_s307["vpd"] >= merged_df_s307["vpd_q90"],
    "vpd_local_q95": merged_df_s307["vpd"] >= merged_df_s307["vpd_q95"],
}

for cond_name, cond_series in conditions_def.items():
    merged_df_s307[cond_name] = cond_series.astype(int)

# S3-07_absolute_humidity_features.csv 저장
feature_save_cols = [
    "analysis_id", "fire_exposure_id", "control_rank", "sample_group", "target",
    "기상셀ID", "기준시각", "기후지형유형", "month", "hour", "date",
    "temp", "rh", "ah", "vpd", "rh_z", "rh_pct", "ah_z", "ah_pct", "vpd_z", "vpd_pct"
]
write_table(merged_df_s307[feature_save_cols], "S3-07_absolute_humidity_features.csv")

# Z-score 분위수 임계치 요약 S3-07_local_percentile_thresholds.csv 저장
thresh_cols = ["rh_q05", "rh_q10", "rh_q20", "ah_q05", "ah_q10", "ah_q20", "vpd_q80", "vpd_q90", "vpd_q95"]
group_cols = ["기상셀ID", "month", "hour"]
threshold_table = hourly_weather_s307[group_cols + thresh_cols].drop_duplicates().reset_index(drop=True)

cell_mapping = merged_df_s307[["기상셀ID", "기후지형유형"]].drop_duplicates().set_index("기상셀ID")["기후지형유형"].to_dict()
threshold_table["기후지형유형"] = threshold_table["기상셀ID"].map(cell_mapping)

percentile_summary = threshold_table.groupby(["기후지형유형", "month"])[thresh_cols].mean().reset_index()
write_table(percentile_summary, "S3-07_local_percentile_thresholds.csv")

BOOT_N_S307 = 500
rng_s307 = np.random.default_rng(307)

def bootstrap_ci_s307(values, blocks, reps=BOOT_N_S307):
    vals = np.asarray(values, dtype=float)
    block_arr = np.asarray(blocks)
    ok = np.isfinite(vals) & pd.notna(block_arr)
    vals = vals[ok]
    block_arr = block_arr[ok]
    if len(vals) < 2: return np.nan, np.nan
    unique_blocks = pd.unique(block_arr)
    if len(unique_blocks) < 2: return np.nan, np.nan
    by_block = {b: vals[block_arr == b] for b in unique_blocks}
    means = np.empty(reps)
    for r in range(reps):
        sample_blocks = rng_s307.choice(unique_blocks, size=len(unique_blocks), replace=True)
        sample_vals = np.concatenate([by_block[b] for b in sample_blocks])
        means[r] = np.nanmean(sample_vals)
    return float(np.nanpercentile(means, 2.5)), float(np.nanpercentile(means, 97.5))

def summarize_binary_conditions_s307(df, cond_list):
    cond_rows = []
    groups = {"전체": df.copy()}
    for ct, sub in df.groupby("기후지형유형"):
        groups[ct] = sub.copy()
        
    for grp_name, df_g in groups.items():
        n_fire = len(df_g[df_g["target"] == 1])
        n_control = len(df_g[df_g["target"] == 0])
        if n_fire == 0 or n_control == 0: continue
        
        for cond in cond_list:
            fire_ok = df_g[df_g["target"] == 1][cond].to_numpy()
            ctrl_df = df_g[df_g["target"] == 0][["fire_exposure_id", cond]].copy()
            ctrl_mean = ctrl_df.groupby("fire_exposure_id")[cond].mean()
            
            fire_df = df_g[df_g["target"] == 1][["fire_exposure_id", "date", ak_cell_col, cond]].copy().set_index("fire_exposure_id")
            
            paired = fire_df.join(ctrl_mean, lsuffix="_fire", rsuffix="_ctrl_mean").dropna()
            diff = paired[f"{cond}_fire"] - paired[f"{cond}_ctrl_mean"]
            
            n = len(paired)
            if n == 0: continue
            
            fire_rate = float(paired[f"{cond}_fire"].mean())
            control_rate = float(paired[f"{cond}_ctrl_mean"].mean())
            rd = float(diff.mean())
            
            lo, hi = bootstrap_ci_s307(diff.to_numpy(), paired["date"].to_numpy())
            
            try: p = float(stats.ttest_1samp(diff, 0, nan_policy="omit").pvalue) if n > 1 else np.nan
            except Exception: p = np.nan
            
            pr = (fire_rate + 1e-9) / (control_rate + 1e-9)
            odds_fire = (fire_rate + 0.5 / n) / (1.0 - fire_rate + 0.5 / n)
            odds_control = (control_rate + 0.5 / n) / (1.0 - control_rate + 0.5 / n)
            or_val = odds_fire / odds_control if odds_control else np.nan
            
            cond_rows.append({
                "group": grp_name,
                "condition": cond,
                "n": int(n),
                "fire_condition_rate": fire_rate,
                "control_condition_rate": control_rate,
                "matched_risk_diff": rd,
                "date_boot_ci_low": lo,
                "date_boot_ci_high": hi,
                "matched_prevalence_ratio": pr,
                "matched_or_approx": or_val,
                "paired_t_p": p,
                "date_n": int(paired["date"].nunique()),
                "cell_n": int(paired[ak_cell_col].nunique()),
                "support_flag": "ok" if (n >= 30 and paired["date"].nunique() >= 10 and paired[ak_cell_col].nunique() >= 5) else "sparse_do_not_interpret"
            })
            
    summary_df = pd.DataFrame(cond_rows)
    summary_df["paired_t_q"] = bh_fdr(summary_df["paired_t_p"])
    return summary_df

print("Summarizing binary thresholds...")
cond_list = list(conditions_def.keys())
thresh_performance_s307 = summarize_binary_conditions_s307(merged_df_s307, cond_list)
write_table(thresh_performance_s307, "S3-07_threshold_performance_summary.csv")

# 임시 파일로도 갱신하여 플롯 생성용 변수로 전달
write_table(merged_df_s307, "S3-07_merged_df_temp.csv.gz", compression="gzip")
print("S3-07 statistics calculation completed!")

### S3-07-3. ECDF 및 임계곡선 시각화

In [41]:
# S3-07 시각화 및 감사
print("Creating S3-07 absolute vs relative ECDF...")
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

fire_df = merged_df_s307[merged_df_s307["target"] == 1].copy()
sns.ecdfplot(
    data=fire_df,
    x="ah",
    hue="기후지형유형",
    linewidth=2.5,
    palette={"영동 해안형": "#4c78a8", "영서 내륙형": "#f58518", "고지·산간형": "#e15759"},
    ax=axes[0]
)
axes[0].set_title("산불 노출지의 절대습도(AH) 원값 ECDF (기후지형유형별)")
axes[0].set_xlabel("절대습도 AH (g/m³)")
axes[0].set_ylabel("누적 비율 (ECDF)")

sns.ecdfplot(
    data=fire_df,
    x="rh_pct",
    hue="기후지형유형",
    linewidth=2.5,
    palette={"영동 해안형": "#4c78a8", "영서 내륙형": "#f58518", "고지·산간형": "#e15759"},
    ax=axes[1]
)
axes[1].set_title("산불 노출지의 국지 상대습도 백분위수(rh_pct) ECDF (기후지형유형별)")
axes[1].set_xlabel("국지 상대습도 백분위수 (rh_pct, 0~1)")
axes[1].set_ylabel("누적 비율 (ECDF)")
axes[1].axvline(0.1, color="gray", linestyle="--", linewidth=1, label="하위 10%선")
axes[1].legend(loc="lower right")

ecdf_path = save_current_plot("S3-07_absolute_vs_relative_ecdf.png")

print("Creating S3-07 threshold curve by type...")
plt.figure(figsize=(10, 6))
sns.lineplot(
    data=percentile_summary,
    x="month",
    y="rh_q10",
    hue="기후지형유형",
    marker="o",
    linewidth=2.5,
    palette={"영동 해안형": "#4c78a8", "영서 내륙형": "#f58518", "고지·산간형": "#e15759"}
)
plt.title("월별 × 기후지형유형별 하위 10% 상대습도 임계치(rh_q10) 변화 곡선")
plt.xlabel("월 (Month)")
plt.ylabel("하위 10% 상대습도 임계치 (%)")
plt.xticks(range(1, 13))
plt.ylim(0, 50)
plt.legend(title="기후지형유형")

curve_path = save_current_plot("S3-07_threshold_curve_by_type.png")

# 매니페스트 저장
plot_paths = [ecdf_path, curve_path]
plot_manifest_s307 = pd.DataFrame([{"plot_path": str(p.relative_to(REPO_ROOT)).replace("\\", "/"), "file_name": p.name} for p in plot_paths])
write_table(plot_manifest_s307, "S3-07_plot_manifest.csv")

artifact_manifest_s307 = pd.DataFrame([
    { "artifact_type": "table", "path": f"jsw/강원_재_EDA/outputs/Step3/tables/{name}" }
    for name in ["S3-07_absolute_humidity_features.csv", "S3-07_local_percentile_thresholds.csv", "S3-07_threshold_performance_summary.csv", "S3-07_plot_manifest.csv"]
] + [
    {"artifact_type": "plot", "path": f"jsw/강원_재_EDA/outputs/Step3/plots/{p.name}"}
    for p in plot_paths
])
write_table(artifact_manifest_s307, "S3-07_artifact_manifest.csv")
print("S3-07 plots and artifact manifest completed!")
thresh_performance_s307[thresh_performance_s307["group"].eq("전체")].sort_values("matched_or_approx", ascending=False)
